# KLTN — 3D-HyperGAMO trên Indian Pines mất cân bằng
## Giao thức B: nhãn nhất quán, development tách riêng, test khóa đến cuối

Notebook này được viết lại từ đầu để thay thế việc vá dần notebook A. Mục tiêu là vừa **chạy thí nghiệm có thể tái lập**, vừa giúp người đọc chưa quen ảnh siêu phổ (HSI) hiểu được từng bước.

### Câu hỏi nghiên cứu của notebook

1. Ba cách xử lý mất cân bằng thông thường — CNN thường, Random Oversampling (ROS), Class Weight — hoạt động thế nào trên cùng kiến trúc bộ phân lớp?
2. 3D-HyperGAMO có lợi gì và có hạn chế gì đối với các lớp rất ít mẫu?
3. Các mẫu sinh có thật sự đa dạng hay đang bám quá sát một vài mẫu thật?
4. Sau khi xác định được hạn chế bằng chẩn đoán, mới thiết kế **một cải tiến duy nhất** và kiểm tra trên cùng giao thức.

### Nguyên tắc

- Giữ **16 lớp**, patch **11×11**, PCA **40 thành phần**.
- Giao thức B dùng train center chỉ khi nhãn DIP và Standard GT nhất quán: **5528 train / 4721 test**.
- Test không dùng để chọn learning rate, epoch, checkpoint hay ý tưởng cải tiến.
- Development validation chỉ phục vụ chọn cấu hình; **không gọi là spatial-independent validation** vì patch vẫn chồng lấn.
- Sau khi chọn cấu hình CNN, mô hình cuối được train lại trên **toàn bộ 5528 mẫu train**.
- HyperGAMO baseline dùng **LR = 0.0002, 8000 classifier updates** cố định, không săn checkpoint đẹp trên validation.
- Mỗi phương pháp train ở **section riêng, chạy tuần tự**, kết quả được lưu để dùng lại.

> **Lưu ý quan trọng:** notebook này không tuyên bố đã loại bỏ hoàn toàn phụ thuộc không gian. Kết quả cuối được diễn giải là hiệu quả trên **fixed within-scene Indian Pines benchmark**.

## 0. Workflow tổng quát

```text
Indian Pines + Standard GT + DIP mask
                 ↓
      kiểm tra nguồn và hash
                 ↓
       Giao thức B nhất quán
      5528 train / 4721 test
                 ↓
      Development split cố định
                 ↓
 PCA/MinMax fit trên development train
                 ↓
   Tuning CNN bằng seed 42 
                 ↓
      khóa LR + số optimizer updates từng CNN
                 ↓
 PCA/MinMax fit lại trên toàn bộ 5528
                 ↓
        FINAL TRAIN tuần tự
 CNN thường → ROS → Class Weight
 → C-real-only matched → HyperGAMO
                 ↓
      chẩn đoán mẫu sinh GAN
                 ↓
     xác định hạn chế thực sự
                 ↓
        1 cải tiến duy nhất
                 ↓
       khóa phương pháp cuối
                 ↓
         mở 4721 test
                 ↓
 bảng số + per-class + confusion matrix
 + bản đồ lỗi + trực quan 2D/3D
```

Các section dưới đây cố ý tách nhỏ. Sau mỗi bước quan trọng có phần **“Kết quả này nói gì?”** để người đọc không phải tự suy luận từ output thô.

### Cách chạy v2

1. Giữ `CHE_DO_CHAY='preflight'`, Run All trên CPU: chỉ kiểm nguồn, split, PCA/cache.
2. Bật GPU, restart kernel, đặt `CHE_DO_CHAY='smoke'`: kiểm các đường cập nhật và freeze.
3. `development`: chọn cấu hình CNN, không train GAN nghiên cứu.
4. Giữ nguyên cấu hình nghiên cứu, chuyển `baseline`: train final và lưu recovery.
5. `diagnostics`: đọc model đã train, không train thêm, không mở test.
6. Chỉ dùng `test` cùng `CHO_PHEP_MO_TEST=True` sau quyết định cải tiến và đủ artifacts.

Sau mỗi phiên Kaggle, **Save Version kèm Save Output**. Phiên mới gắn Output
trước đó vào Input; code tự khôi phục run cùng config về `/kaggle/working`.
Nếu có nhiều Output cùng run, điền `RESUME_RUN_PATH`. Không đổi config để tiếp tục cùng run.


In [ ]:
# ============================================================
# 1. CẤU HÌNH DUY NHẤT CỦA NOTEBOOK
# ============================================================
# Mục tiêu: mọi thông số quan trọng chỉ nằm ở một nơi.
# Markdown và các hàm phía sau đọc từ cấu hình này, tránh tình trạng
# tài liệu ghi một kiểu nhưng code chạy một kiểu.

from pathlib import Path
import os, sys, json, hashlib, shutil, zipfile, platform, subprocess, time, math

CAU_HINH = {
    # Dữ liệu / giao thức
    "ten_giao_thuc": "B_label_consistent_16class_patch11_pca40_v2_reviewed",
    "so_lop": 16,
    "patch_size": 11,
    "pca_components": 40,
    "pca_solver": "full",
    "development_seed": 42,
    "development_val_fraction": 0.20,

    # Seed final: cùng master split, khác stochastic training
    "seeds": [42, 43, 44],

    # CNN
    "cnn_batch_size": 32,
    "cnn_learning_rates": [1e-3, 5e-4, 2e-4, 1e-4],
    "cnn_tuning_max_epochs": 100,
    "cnn_tuning_patience": 15,
    "cnn_min_delta_nll": 1e-6,

    # 3D-HyperGAMO / matched classifier
    "gan_batch_size": 100,
    "gan_learning_rate": 2e-4,
    "gan_beta1": 0.5,
    "gan_classifier_updates": 8000,
    "gan_log_every": 500,
    "gan_latent_dim": 100,

    # Chẩn đoán GAN
    "diagnostic_generated_per_class": 512,
    "diagnostic_saved_per_class": 32,

    # Kỹ thuật
    "eval_batch_size": 64,
    "gpu_index": 0,
}

# Không tự mở test. Khi baseline + cải tiến đã khóa mới đổi thành True.
CHO_PHEP_MO_TEST = False

# Nếu chạy local, có thể sửa thư mục này.
DUONG_DAN_DU_LIEU_LOCAL = Path("./dataset")

# Kaggle ưu tiên /kaggle/working; local dùng thư mục cạnh notebook.
THU_MUC_KET_QUA = (
    Path("/kaggle/working")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "ket_qua_protocol_B"
)
THU_MUC_KET_QUA.mkdir(parents=True, exist_ok=True)

print(json.dumps(CAU_HINH, ensure_ascii=False, indent=2))
print("CHO_PHEP_MO_TEST =", CHO_PHEP_MO_TEST)
print("Thư mục kết quả =", THU_MUC_KET_QUA)
# Chạy theo giai đoạn; Run All mặc định chỉ chuẩn bị/kiểm dữ liệu CPU.
CHE_DO_CHAY = 'preflight'  # preflight / smoke / development / baseline / diagnostics / test
assert CHE_DO_CHAY in {'preflight', 'smoke', 'development', 'baseline', 'diagnostics', 'test'}
REQUIRE_GPU_FOR_TRAIN = True
IMPLEMENTATION_SHA256 = '5e15f8214bf273bb889187b96c80407a8000e5e1a3ef30734bcfa86ced275a6f'
# Điền đường dẫn nếu cần. Các alias GT cùng nội dung được nhận diện bằng hash.
HSI_PATH_EXPLICIT = None
GT_PATH_EXPLICIT = None
RESUME_RUN_PATH = None  # thư mục run trong Kaggle Input nếu có nhiều bản cùng run_id
# Chỉ điền sau diagnostics: xem section 22. Không dùng {} để mở test.
PHUONG_PHAP_CAI_TIEN = None
print('Chế độ chạy:', CHE_DO_CHAY)


### Giải thích cấu hình

- `development_seed=42`: chỉ dùng để tạo development split và tuning CNN. Seed 43/44 **không được tự chọn checkpoint**.
- `cnn_learning_rates`: giữ đúng search space đã dùng trước đây; không mở rộng sau khi xem kết quả để tránh cherry-picking.
- `gan_classifier_updates=8000`: giữ ngân sách baseline 3D-HyperGAMO cố định.
- `CHO_PHEP_MO_TEST=False`: test được khóa mặc định.

**Kết quả này nói gì?** Đây là “hợp đồng thí nghiệm”. Nếu sau này đổi một thông số quan trọng thì phải sinh một run mới, không ghi đè run cũ.
Các chế độ: `preflight` chỉ CPU dữ liệu; `smoke` kiểm cập nhật nhỏ trên GPU; `development` tuning CNN trên GPU;
`baseline` final CNN/C-real/GAN; `diagnostics` chẩn đoán đã train;
`test` chỉ khi registry cuối đã khóa. Tất cả giữ cùng cấu hình nghiên cứu để tái dùng run.


In [ ]:
# ============================================================
# 2. THƯ VIỆN VÀ HÀM LƯU KẾT QUẢ
# ============================================================
# Phần này ưu tiên tiếng Việt trong chú thích; tên API kỹ thuật của thư viện
# vẫn giữ nguyên vì đó là tên chuẩn bắt buộc.

import base64, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.ndimage import maximum_filter
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    precision_recall_fscore_support, f1_score
)
import joblib
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

TEN_LOP = [
    "Alfalfa", "Corn-notill", "Corn-mintill", "Corn", "Grass-pasture",
    "Grass-trees", "Grass-pasture-mowed", "Hay-windrowed", "Oats",
    "Soybean-notill", "Soybean-mintill", "Soybean-clean", "Wheat",
    "Woods", "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"
]
ID_LOP = np.arange(1, 17)

# Mapping của mask DIP nguồn tác giả -> ID lớp chuẩn 1..16.
DIP_MAPPING = np.array([0, 2, 3, 5, 6, 8, 10, 11, 12, 14, 1, 4, 7, 9, 13, 15, 16])

# Count B đã được audit độc lập trước khi viết notebook này.
B_TRAIN_MONG_DOI = np.array([21,753,426,138,209,376,16,228,10,469,1390,311,125,720,287,49])
B_TEST_MONG_DOI  = np.array([25,675,404,99,274,354,12,250,10,503,1065,282,80,545,99,44])

FIGURES = {}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(4 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()


def luu_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def doc_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def luu_hinh(fig, ten, thu_muc=None, dpi=160):
    """Hiển thị và lưu cùng một hình để có thể dùng lại trong báo cáo."""
    thu_muc = Path(thu_muc or (RUN / "figures" if "RUN" in globals() else THU_MUC_KET_QUA / "figures_tam"))
    thu_muc.mkdir(parents=True, exist_ok=True)
    path = thu_muc / f"{ten}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return path


def tim_file_du_lieu(filename, explicit=None):
    """Tìm đúng một file; không âm thầm chọn khi có nhiều bản sao."""
    if explicit:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p

    candidates = []
    if Path("/kaggle/input").is_dir():
        candidates.extend(p for p in Path("/kaggle/input").rglob("*.mat") if p.name.lower() == filename.lower())
    local = DUONG_DAN_DU_LIEU_LOCAL / filename
    if local.is_file():
        candidates.append(local)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise RuntimeError(f"Cần đúng 1 file {filename}, nhưng tìm được {len(candidates)}: {candidates}")
    return candidates[0]


def stretch(a):
    lo, hi = np.percentile(a, [2, 98])
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)


def trich_patch(cube, coords, patch_size):
    """Cắt patch quanh tâm. Padding 0 ở biên ảnh."""
    r = patch_size // 2
    padded = np.pad(cube, ((r, r), (r, r), (0, 0)), mode="constant")
    out = np.empty((len(coords), patch_size, patch_size, cube.shape[-1], 1), dtype=np.float32)
    for i, (y, x) in enumerate(coords):
        out[i, ..., 0] = padded[y:y+patch_size, x:x+patch_size]
    return out


def bao_cao_metrics(y_true, prob, so_lop=16):
    pred = np.asarray(prob).argmax(1)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, pred, labels=np.arange(so_lop), zero_division=0
    )
    gm = float(np.exp(np.log(recall).mean())) if np.all(recall > 0) else 0.0
    summary = {
        "OA": float(accuracy_score(y_true, pred)),
        "AA": float(recall.mean()),
        "Macro_F1": float(f1.mean()),
        "GM": gm,
        "Kappa": float(cohen_kappa_score(y_true, pred)),
    }
    per_class = pd.DataFrame({
        "class_id": ID_LOP,
        "class_name": TEN_LOP,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support.astype(int),
    })
    return summary, per_class, pred


def balanced_nll(y_true, prob, eps=1e-7):
    """Negative log-likelihood cân bằng theo lớp: mỗi lớp đóng góp ngang nhau."""
    prob = np.clip(np.asarray(prob), eps, 1.0)
    loss = -np.log(prob[np.arange(len(y_true)), y_true])
    class_loss = []
    for c in range(len(ID_LOP)):
        idx = (y_true == c)
        if idx.any():
            class_loss.append(float(loss[idx].mean()))
    return float(np.mean(class_loss))

print("Python:", platform.python_version())
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
def immutable_json(path, obj):
    path = Path(path)
    if path.exists():
        if doc_json(path) != obj:
            raise RuntimeError(f'Artifact đã có nội dung khác: {path}. Dùng run mới.')
    else:
        luu_json(path, obj)

def require_gpu_training():
    if REQUIRE_GPU_FOR_TRAIN and not tf.config.list_logical_devices('GPU'):
        raise RuntimeError('Huấn luyện cần GPU. Bật accelerator; CPU chỉ dùng preflight/smoke nhỏ.')
    if REQUIRE_GPU_FOR_TRAIN:
        smoke = RUN / 'hypergamo_smoke_pass.json'
        record = doc_json(smoke) if smoke.exists() else {}
        if not record.get('passed') or not record.get('checkpoint_roundtrip') or record.get('run_id') != run_id or record.get('implementation') != IMPLEMENTATION_SHA256 or record.get('tensorflow') != tf.__version__ or not record.get('gpu_present'):
            raise RuntimeError('Cần chạy CHE_DO_CHAY=smoke trên GPU và cùng runtime trước development/baseline.')

def array_sha(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.shape).encode()); h.update(a.dtype.str.encode()); h.update(a.tobytes())
    return h.hexdigest()

def cache_contract(**extra):
    return {'run_id': run_id, 'implementation': IMPLEMENTATION_SHA256, **extra}

def save_npz_checked(path, contract, **arrays):
    path = Path(path)
    meta_path = path.with_suffix(path.suffix + '.json')
    if path.exists():
        if not meta_path.exists():
            raise RuntimeError(f'Cache thiếu provenance: {path}')
        meta = doc_json(meta_path)
        if meta['contract'] != contract or meta['sha256'] != sha256_file(path):
            raise RuntimeError(f'Cache sai cấu hình/nội dung: {path}')
        return
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    tmp.replace(path)
    luu_json(meta_path, {'contract': contract, 'sha256': sha256_file(path)})

def verify_npz(path, contract):
    meta = doc_json(Path(path).with_suffix(Path(path).suffix + '.json'))
    if meta['contract'] != contract or meta['sha256'] != sha256_file(path):
        raise RuntimeError(f'Cache không khớp: {path}')

def load_done_checked(folder, required_weights):
    folder = Path(folder); path = folder / 'done.json'
    if not path.exists():
        return None
    meta = doc_json(path)
    if not meta.get('complete') or meta.get('run_id') != run_id or meta.get('implementation') != IMPLEMENTATION_SHA256:
        raise RuntimeError(f'Done marker sai contract: {path}')
    hashes = meta.get('artifact_hashes', {})
    for rel in required_weights:
        p = folder / rel
        if not p.is_file() or hashes.get(rel) != sha256_file(p):
            raise RuntimeError(f'Weights thiếu/đã đổi: {p}')
    return meta

def stamp_done(folder, meta, weights_paths):
    folder = Path(folder)
    hashes = {}
    for item in weights_paths:
        p = Path(item)
        if not p.is_absolute():
            p = folder / p
        hashes[p.relative_to(folder).as_posix()] = sha256_file(p)
    result = dict(meta, complete=True, run_id=run_id,
                  implementation=IMPLEMENTATION_SHA256, artifact_hashes=hashes)
    luu_json(folder / 'done.json', result)
    return result

def assert_finite_model(model):
    if not all(np.isfinite(w).all() for w in model.get_weights()):
        raise FloatingPointError('Model chứa NaN/Inf; không ghi complete.')

def save_prep_checked(path, coords):
    coords_hash = array_sha(coords)
    contract = cache_contract(fit_coords_sha256=coords_hash,
                              pca_components=CAU_HINH['pca_components'],
                              pca_solver=CAU_HINH['pca_solver'])
    path = Path(path); sidecar = path.with_suffix('.joblib.json')
    fit_spectra = hsi[coords[:, 0], coords[:, 1]].astype(np.float64)
    if path.exists():
        meta = doc_json(sidecar)
        if meta['contract'] != contract or meta['sha256'] != sha256_file(path):
            raise RuntimeError(f'Preprocessing sai provenance: {path}')
        prep = joblib.load(path)
    else:
        pca = PCA(n_components=CAU_HINH['pca_components'], svd_solver=CAU_HINH['pca_solver'])
        features = pca.fit_transform(fit_spectra)
        scaler = MinMaxScaler(feature_range=(-1, 1)).fit(features)
        prep = {'pca': pca, 'scaler': scaler, 'fit_count': len(coords), 'fit_coords_sha256': coords_hash}
        tmp = path.with_suffix('.joblib.tmp'); joblib.dump(prep, tmp); tmp.replace(path)
        luu_json(sidecar, {'contract': contract, 'sha256': sha256_file(path)})
    assert prep['fit_count'] == len(coords) == prep['pca'].n_samples_
    assert prep['fit_coords_sha256'] == coords_hash
    np.testing.assert_allclose(prep['pca'].mean_, fit_spectra.mean(0), rtol=1e-7, atol=1e-7)
    # Verify scaler was fitted only on these transformed training spectra.
    features = prep['pca'].transform(fit_spectra)
    np.testing.assert_allclose(prep['scaler'].data_min_, features.min(0), rtol=1e-6, atol=1e-6)
    np.testing.assert_allclose(prep['scaler'].data_max_, features.max(0), rtol=1e-6, atol=1e-6)
    return prep


## 3. Đọc dữ liệu gốc và kiểm tra nguồn

Indian Pines corrected có kích thước **145×145×200**: hai trục đầu là vị trí không gian, trục cuối là 200 dải phổ đã loại các dải nhiễu theo bản corrected.

Standard GT có kích thước **145×145**. Mỗi pixel có một nhãn từ 0 đến 16; nhãn `0` nghĩa là pixel không nằm trong tập nhãn chuẩn dùng để đánh giá.

In [ ]:
def tim_file_du_lieu_hash(names, expected_hash, explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        roots = [Path('/kaggle/input'), DUONG_DAN_DU_LIEU_LOCAL,
                 Path(r'C:\Users\ADMIN\OneDrive\Desktop\reproduct - 3dhyper\3D-HyperGAMO-main\3D-HyperGAMO-main\HSI-datasets\Classification')]
        names = {name.lower() for name in names}
        candidates = sorted({p.resolve() for root in roots if root.is_dir()
                             for p in root.rglob('*.mat') if p.name.lower() in names})
    matches = [p for p in candidates if p.is_file() and sha256_file(p) == expected_hash]
    if not matches:
        raise RuntimeError(f'Không tìm được file nguồn có hash đã audit. Candidates: {candidates}. Điền đường dẫn explicit.')
    # Multiple identical copies are harmless; require identical content, not arbitrary first file.
    return sorted(matches)[0]

# ============================================================
# 3.1 ĐỌC HSI, STANDARD GT VÀ DIP MASK
# ============================================================

HSI_PATH = tim_file_du_lieu_hash(["indian_pines_corrected.mat"], "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939", HSI_PATH_EXPLICIT)
GT_PATH = tim_file_du_lieu_hash(["indian_pines_gt.mat", "indian_pines_corrected_gt.mat"], "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c", GT_PATH_EXPLICIT)

hsi = loadmat(HSI_PATH)["indian_pines_corrected"].astype(np.float32)
gt = loadmat(GT_PATH)["indian_pines_gt"].astype(np.int16)

assert hsi.shape == (145, 145, 200)
assert gt.shape == (145, 145)
assert np.array_equal(np.unique(gt), np.arange(17))
assert int((gt > 0).sum()) == 10249
assert np.isfinite(hsi).all()

# DIP mask được nhúng trực tiếp để notebook không phụ thuộc link tải ngoài.
DIP_B64 = "eJzt201vmzAYwHHDWvKyjeS6G7tvUxupk5JbutOkRZq23KtIoRpTl0RNJvVr7BuPBAjEL/AYG9sk/kuozQvwkwvFNOpsOv82vQ/uPt0Es+n842P0FAbfnxa7x/Xzn0mwWW+jlw/Bl+dwsQuXwXo1Cea//gY/w00w+hzcjiej8eRuFIxubseoRs7XmR9/efmBkBd/7cZLL33pOn38L12ceHkXL9FqGS1Wm2gVbh+W0fb3OlrtHpbbcJeu58bL+3h77r46JqxXSeIbciWJJIPERbJBLNHrNOzpbtzJBuSDGCKNoEzkHMJBmEgfKBedoAwB5apLAhFhIEJ08SAHB+Ei/SAEAiH5INcwUHqeUUCUFIBcs0CF30RGgFw9ID/fAo2TPVsEsSYCUkA+A4RfPEgQKZIB8hkgVxPIZ4BcAoRIECEqgLI4QX4joLf7aoF8HhAiQbiIAuLKZ4O05MNAfAfl1Wn0N3VoiYAcJ7+UXJUmE0ScMHRQuUgeiHIGawF5SYkIBioVoWRXhoGKOxUDecfKQKVR910LhCyIDUqeAB9DqkCHMz/JOFAXsykEFeMAkRvPpis8oNMYswmqiBwhJqh+PNOPFoAov5EtCNuhetBwaBJouK8UhBSChmmmgIYwEDJthHBQgwFBlMsE8wVK2R9IIbddZoIKj1lXe6Wg4o2RdhDaD0/hcs8CUachYBBvrQTRpiFtA3E7+EC0O1W5DFGQ9HHhBQFn9QpB6mLdNWrLgqo6dxDwz5EAkCSRBSkCycHUBg0GBoEGh4wBDbLkCcRqF8irCh1nIYyJAf8RLQbyeEEcI0QTQUCIA1TNEQblH0UAQBBPOQggqwDxNwCJAENkQVUiJaDy47lxECkCgTwZoF6cGaBeHgDEUy1Qr9cukJDIgrSAREQtApVMQSxIGCRDZAyoThSQjPp9paDSN/eTzAEh40BIH8ihlYo0gKic5Bw3DmTUWSaYJBA5RhZ0yaDqz9wUH9RQEHnvfNqbOP7PD2GgDmWltoJ4RQKgqowDofR/XxoBdciVwCA+0TmBKn+yjYI6xEpwEJeoWVCNKCDgShJ2fr6gJqsLuqanDcTwaAOxPPJAfCsxPQ2DWGPH9kgDlWLxd5R4FIIK72kWVI0lx0gv6PidASCbzWaz2Ww2pf0H3l+xCA=="
dip_bytes = __import__("zlib").decompress(base64.b64decode(DIP_B64))
encoded_dip = loadmat(io.BytesIO(dip_bytes))["indianpines_disjoint_dset"].astype(np.int16)
assert encoded_dip.shape == gt.shape

source_hashes = {
    "hsi_sha256": sha256_file(HSI_PATH),
    "gt_sha256": sha256_file(GT_PATH),
    "dip_sha256": hashlib.sha256(dip_bytes).hexdigest(),
}

# Hash đã kiểm chứng trong notebook A. Nếu khác thì dừng để tránh dùng nhầm file.
assert source_hashes["hsi_sha256"] == "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939"
assert source_hashes["gt_sha256"] == "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c"
assert source_hashes["dip_sha256"] == "430c37ce56a160d410378aa136e5e696e2436e2fe03e275f306c3502f622e77b"

print("HSI:", HSI_PATH, hsi.shape)
print("GT :", GT_PATH, gt.shape)
print("Số pixel có nhãn chuẩn:", int((gt > 0).sum()))
print("SHA256:")
print(json.dumps(source_hashes, indent=2))


**Kết quả này nói gì?** Nếu ba hash khớp, ta biết notebook đang dùng đúng bộ ảnh, Standard GT và DIP mask đã audit trước đó. Đây là lớp bảo vệ đầu tiên chống chạy nhầm dữ liệu.

In [ ]:
# ============================================================
# 3.2 TRỰC QUAN HSI: ẢNH GIẢ MÀU, GT, PHÂN BỐ LỚP
# ============================================================

class_counts_all = pd.DataFrame({
    "class_id": ID_LOP,
    "class_name": TEN_LOP,
    "total": [int((gt == c).sum()) for c in ID_LOP],
})

pseudo_rgb = np.stack([stretch(hsi[..., b]) for b in [30, 20, 10]], axis=-1)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].imshow(pseudo_rgb)
ax[0].set_title("Ảnh giả màu từ 3 dải phổ")
ax[0].axis("off")

im = ax[1].imshow(gt, cmap="tab20", vmin=0, vmax=16)
ax[1].set_title("Standard GT — 16 lớp")
ax[1].axis("off")

ax[2].bar(class_counts_all["class_id"], class_counts_all["total"])
ax[2].set_xticks(ID_LOP)
ax[2].set_xlabel("ID lớp")
ax[2].set_ylabel("Số pixel có nhãn")
ax[2].set_title("Mất cân bằng trên toàn Standard GT")
fig.tight_layout()
luu_hinh(fig, "01_hsi_gt_phan_bo_lop")

display(class_counts_all)
print("Lớp ít nhất / nhiều nhất:", class_counts_all.total.min(), "/", class_counts_all.total.max())
print("Tỷ lệ max/min:", round(class_counts_all.total.max() / class_counts_all.total.min(), 2))

**Cách đọc hình:** ảnh giả màu chỉ là cách nhìn trực quan bằng 3 dải phổ; mô hình không nhận ảnh RGB này. Mỗi pixel thực tế có 200 giá trị phổ. Biểu đồ bên phải cho thấy chênh lệch rất lớn giữa các lớp — đây chính là động cơ của bài toán mất cân bằng.

In [ ]:
# ============================================================
# 3.3 KHỐI HSI 3D — CHỈ ĐỂ HIỂU CẤU TRÚC DỮ LIỆU
# ============================================================

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
yy, xx = np.mgrid[0:hsi.shape[0]:4, 0:hsi.shape[1]:4]
for band in [0, 30, 60, 90, 120, 150, 199]:
    face = plt.get_cmap("viridis")(stretch(hsi[..., band])[::4, ::4])
    ax.plot_surface(xx, yy, np.full_like(xx, band), facecolors=face,
                    rstride=1, cstride=1, shade=False, alpha=.75)
ax.set_xlabel("Cột")
ax.set_ylabel("Hàng")
ax.set_zlabel("Chỉ số dải phổ")
ax.set_title("Khối ảnh siêu phổ: 2 chiều không gian + 1 chiều phổ")
ax.view_init(elev=20, azim=-55)
fig.tight_layout()
luu_hinh(fig, "02_khoi_hsi_3d")

**Kết quả này nói gì?** Trục thứ ba trong hình là **dải phổ**, không phải chiều cao địa hình. Một vị trí `(hàng, cột)` có cả một vector phổ; do đó HSI chứa nhiều thông tin hơn ảnh RGB thông thường.

In [ ]:
# ============================================================
# 3.4 CHỮ KÝ PHỔ TRUNG BÌNH THEO LỚP
# ============================================================

fig, ax = plt.subplots(figsize=(13, 5))
for cid in ID_LOP:
    ax.plot(hsi[(gt == cid) & (DIP_MAPPING[encoded_dip] == gt) & (DIP_MAPPING[encoded_dip] > 0)].mean(axis=0), label=f"{cid}: {TEN_LOP[cid-1]}")
ax.set_xlabel("Chỉ số dải phổ")
ax.set_ylabel("Giá trị trung bình")
ax.set_title("Đường phổ trung bình của B train — 16 lớp")
ax.grid(alpha=.2)
ax.legend(ncol=2, fontsize=7)
fig.tight_layout()
luu_hinh(fig, "03_pho_trung_binh_16_lop")


**Cách đọc hình:** mỗi đường là trung bình 200 dải của một lớp. Các lớp có thể giống nhau ở một số dải nhưng khác ở các dải khác. 3D CNN khai thác đồng thời cấu trúc không gian của patch và thông tin phổ này.

## 4. Giao thức B — sửa bất đồng nhãn của giao thức A

Giao thức A giữ toàn bộ train center của DIP mask, kể cả 161 vị trí không nhất quán với Standard GT. Giao thức B chỉ giữ một train center khi:

\[
DIP>0 \land GT>0 \land label_{DIP}=label_{GT}
\]

Mục tiêu của thay đổi này **không phải chữa overlap**, mà là làm cho train và test cùng dựa trên một hệ nhãn thống nhất.

In [ ]:
# ============================================================
# 4.1 DỰNG A ĐỂ ĐỐI CHIẾU, SAU ĐÓ DỰNG B
# ============================================================

author_train = DIP_MAPPING[encoded_dip].astype(np.int16)
a_test_labels = np.where(author_train == 0, gt, 0).astype(np.int16)

a_train_mask = author_train > 0
a_test_mask = a_test_labels > 0

# B: chỉ giữ vị trí DIP train nếu Standard GT có nhãn và hai nguồn đồng ý nhãn.
b_train_mask = (author_train > 0) & (gt > 0) & (author_train == gt)
b_test_mask = (gt > 0) & (~b_train_mask)

assert int(a_train_mask.sum()) == 5689
assert int(a_test_mask.sum()) == 4711
assert int(b_train_mask.sum()) == 5528
assert int(b_test_mask.sum()) == 4721
assert not np.any(b_train_mask & b_test_mask)
assert np.all((b_train_mask | b_test_mask) == (gt > 0))

b_train_counts = np.array([(b_train_mask & (gt == c)).sum() for c in ID_LOP])
b_test_counts = np.array([(b_test_mask & (gt == c)).sum() for c in ID_LOP])
np.testing.assert_array_equal(b_train_counts, B_TRAIN_MONG_DOI)
np.testing.assert_array_equal(b_test_counts, B_TEST_MONG_DOI)

removed = a_train_mask & (~b_train_mask)
removed_bg = a_train_mask & (gt == 0)
removed_wrong_nonzero = a_train_mask & (gt > 0) & (author_train != gt)

comparison_rows = []
for cid in ID_LOP:
    comparison_rows.append({
        "class_id": cid,
        "class_name": TEN_LOP[cid-1],
        "A_train": int((a_train_mask & (author_train == cid)).sum()),
        "B_train": int((b_train_mask & (gt == cid)).sum()),
        "A_test": int((a_test_mask & (a_test_labels == cid)).sum()),
        "B_test": int((b_test_mask & (gt == cid)).sum()),
        "removed_from_A_train": int((removed & (author_train == cid)).sum()),
    })
comparison_ab = pd.DataFrame(comparison_rows)
comparison_ab["delta_train"] = comparison_ab.B_train - comparison_ab.A_train
comparison_ab["delta_test"] = comparison_ab.B_test - comparison_ab.A_test

display(comparison_ab)
print("A train/test:", int(a_train_mask.sum()), int(a_test_mask.sum()))
print("B train/test:", int(b_train_mask.sum()), int(b_test_mask.sum()))
print("161 vị trí bị ảnh hưởng =", int(removed.sum()))
print("  - Standard GT = 0:", int(removed_bg.sum()))
print("  - Khác nhãn nhưng GT > 0:", int(removed_wrong_nonzero.sum()))
print("Imbalance A:", round(comparison_ab.A_train.max()/comparison_ab.A_train.min(), 2))
print("Imbalance B:", round(comparison_ab.B_train.max()/comparison_ab.B_train.min(), 2))

In [ ]:
# ============================================================
# 4.2 TRỰC QUAN A -> B THEO LỚP
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(16)
w = 0.38
ax.bar(x - w/2, comparison_ab.A_train, width=w, label="A train")
ax.bar(x + w/2, comparison_ab.B_train, width=w, label="B train")
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in ID_LOP])
ax.set_xlabel("ID lớp")
ax.set_ylabel("Số mẫu train")
ax.set_title("Ảnh hưởng của bộ lọc nhãn nhất quán: A so với B")
ax.legend()
ax.grid(axis="y", alpha=.2)
fig.tight_layout()
luu_hinh(fig, "04_A_vs_B_train_counts")

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(comparison_ab.class_id, comparison_ab.removed_from_A_train)
ax.set_xticks(ID_LOP)
ax.set_xlabel("ID lớp theo nhãn DIP ở A")
ax.set_ylabel("Số vị trí bị loại khỏi train B")
ax.set_title("161 vị trí A không còn thuộc train B")
ax.grid(axis="y", alpha=.2)
fig.tight_layout()
luu_hinh(fig, "05_161_vi_tri_bi_anh_huong")

In [ ]:
# ============================================================
# 4.3 BẢN ĐỒ VỊ TRÍ 161 ĐIỂM BỊ ẢNH HƯỞNG
# ============================================================

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].imshow(gt, cmap="tab20", vmin=0, vmax=16)
ax[0].set_title("Standard GT")
ax[0].axis("off")

ax[1].imshow(gt > 0, cmap="gray")
y, x = np.where(removed_bg)
ax[1].scatter(x, y, s=8)
ax[1].set_title(f"151 vị trí DIP train nhưng GT=0")
ax[1].axis("off")

ax[2].imshow(gt > 0, cmap="gray")
y, x = np.where(removed_wrong_nonzero)
ax[2].scatter(x, y, s=25)
ax[2].set_title("10 vị trí DIP và GT khác class")
ax[2].axis("off")
fig.tight_layout()
luu_hinh(fig, "06_ban_do_161_bat_dong_nhan")

**Kết quả này nói gì?** B làm train giảm từ 5689 xuống 5528, nhưng giữ nguyên 10 train mẫu Oats và 16 train mẫu Grass-pasture-mowed. Mức mất cân bằng vẫn rất lớn. Điểm thay đổi đáng chú ý nhất ở test là Grass-pasture-mowed tăng từ 2 lên 12 mẫu, nên recall của lớp cực hiếm này bớt rời rạc hơn.

**Không được suy ra:** B không làm patch hết chồng lấn và không tự làm validation đáng tin hơn.

In [ ]:
# ============================================================
# 4.4 TẠO RUN ID VÀ LƯU MASTER SPLIT B
# ============================================================

protocol_spec = {
    "protocol": CAU_HINH["ten_giao_thuc"],
    "scientific_config": CAU_HINH,
    "implementation_sha256": IMPLEMENTATION_SHA256,
    "cnn_budget_transfer": "freeze_development_optimizer_updates",
    "cnn_selection": "min_balanced_NLL; max_MacroF1; earlier_update",
    "source_hashes": source_hashes,
    "patch_size": CAU_HINH["patch_size"],
    "pca_components": CAU_HINH["pca_components"],
    "class_ids": ID_LOP.tolist(),
    "b_train_counts": b_train_counts.tolist(),
    "b_test_counts": b_test_counts.tolist(),
    "master_train_total": int(b_train_mask.sum()),
    "locked_test_total": int(b_test_mask.sum()),
    "label_rule": "DIP>0 AND GT>0 AND DIP_label==GT",
    "development_seed": CAU_HINH["development_seed"],
    "development_val_fraction": CAU_HINH["development_val_fraction"],
    "test_policy": "locked_until_all_models_and_improvement_are_frozen",
    "spatial_claim": "within_scene; patch overlap is audited and not claimed leakage-free",
}

run_id = hashlib.sha256(json.dumps(protocol_spec, sort_keys=True).encode()).hexdigest()[:12]
RUN = THU_MUC_KET_QUA / f"KLTN_HyperGAMO_B_{run_id}"
if not RUN.exists():
    resume_candidates = [Path(RESUME_RUN_PATH)] if RESUME_RUN_PATH else [
        p.parent for p in Path('/kaggle/input').rglob('manifest.json')
        if p.parent.name == RUN.name] if Path('/kaggle/input').is_dir() else []
    if len(resume_candidates) > 1:
        raise RuntimeError('Có nhiều Output cùng run_id; điền RESUME_RUN_PATH để chọn phiên cần tiếp tục.')
    if resume_candidates:
        source_run = resume_candidates[0]
        if not source_run.is_dir() or doc_json(source_run/'protocol_spec.json') != protocol_spec:
            raise RuntimeError('Resume source không khớp protocol/config/source v2.')
        shutil.copytree(source_run, RUN)
        print('Đã khôi phục Output về working:', RUN)
RUN.mkdir(parents=True, exist_ok=True)
(RUN / "figures").mkdir(exist_ok=True)
(RUN / "tables").mkdir(exist_ok=True)
(RUN / "development").mkdir(exist_ok=True)
(RUN / "final_data").mkdir(exist_ok=True)
(RUN / "models").mkdir(exist_ok=True)
(RUN / "diagnostics").mkdir(exist_ok=True)
(RUN / "test_results").mkdir(exist_ok=True)

master_split = np.full(gt.shape, -1, dtype=np.int8)
master_split[b_train_mask] = 0
master_split[b_test_mask] = 2
coords_master_train = np.argwhere(b_train_mask).astype(np.int16)
coords_locked_test = np.argwhere(b_test_mask).astype(np.int16)

save_npz_checked(
    RUN / "B_master_split.npz", cache_contract(kind="master_split"),
    split=master_split,
    standard_gt=gt,
    coords_train=coords_master_train,
    coords_test=coords_locked_test,
    class_ids=ID_LOP,
)
comparison_ab.to_csv(RUN / "tables" / "A_vs_B_counts.csv", index=False)
immutable_json(RUN / "protocol_spec.json", protocol_spec)

manifest = {
    "run_id": run_id,
    "protocol_spec_sha256": sha256_file(RUN / "protocol_spec.json"),
    "master_split_sha256": sha256_file(RUN / "B_master_split.npz"),
}
immutable_json(RUN / "manifest.json", manifest)
print("RUN =", RUN)
print(json.dumps(manifest, indent=2))


## 5. Development split — chỉ để chọn cấu hình CNN

Master test **4721 mẫu** đã được xác định từ bước trước. Từ đây không thay đổi test theo model score.

Development split được lấy **chỉ từ 5528 master train**. Vì Indian Pines có các vùng lớp rất nhỏ, strict 3-way zero-overlap với patch 11×11 đã được audit là bất khả thi. Do đó validation ở đây chỉ là tín hiệu phát triển mô hình, không phải bằng chứng spatial generalization.

In [ ]:
# ============================================================
# 5.1 TẠO DEVELOPMENT TRAIN / VALIDATION CỐ ĐỊNH
# ============================================================

rng = np.random.default_rng(CAU_HINH["development_seed"])
dev_train_mask = b_train_mask.copy()
dev_val_mask = np.zeros_like(b_train_mask, dtype=bool)

for cid in ID_LOP:
    flat_idx = np.flatnonzero((b_train_mask & (gt == cid)).ravel())
    n_val = max(2, int(round(len(flat_idx) * CAU_HINH["development_val_fraction"])))
    if n_val >= len(flat_idx):
        n_val = len(flat_idx) - 1
    chosen = rng.permutation(flat_idx)[:n_val]
    rr, cc = np.unravel_index(chosen, gt.shape, order="C")
    dev_val_mask[rr, cc] = True
    dev_train_mask[rr, cc] = False

assert len(np.argwhere(dev_train_mask)) == 4422
assert len(np.argwhere(dev_val_mask)) == 1106
assert not np.any(dev_train_mask & dev_val_mask)
assert np.all((dev_train_mask | dev_val_mask) == b_train_mask)

coords_dev_train = np.argwhere(dev_train_mask).astype(np.int16)
coords_dev_val = np.argwhere(dev_val_mask).astype(np.int16)

split_dev = np.full(gt.shape, -1, dtype=np.int8)
split_dev[dev_train_mask] = 0
split_dev[dev_val_mask] = 1
split_dev[b_test_mask] = 2

rows = []
for cid in ID_LOP:
    rows.append({
        "class_id": cid,
        "class_name": TEN_LOP[cid-1],
        "master_train": int((b_train_mask & (gt == cid)).sum()),
        "dev_train": int((dev_train_mask & (gt == cid)).sum()),
        "dev_val": int((dev_val_mask & (gt == cid)).sum()),
        "locked_test": int((b_test_mask & (gt == cid)).sum()),
    })
dev_counts = pd.DataFrame(rows)

display(dev_counts)
print("Development train / val / locked test:", len(coords_dev_train), len(coords_dev_val), len(coords_locked_test))

save_npz_checked(
    RUN / "development" / "dev_split.npz", cache_contract(kind="development_split", rounding="round_per_class_min2"),
    split=split_dev,
    coords_train=coords_dev_train,
    coords_val=coords_dev_val,
    standard_gt=gt,
)
dev_counts.to_csv(RUN / "tables" / "development_counts.csv", index=False)


In [ ]:
# ============================================================
# 5.2 AUDIT CHỒNG LẤN PATCH
# ============================================================

def audit_overlap(split, patch_size):
    footprint = [
        maximum_filter((split == k).astype(np.uint8), size=patch_size, mode="constant", cval=0) > 0
        for k in range(3)
    ]
    names = ["train", "validation", "test"]
    out = {}
    for a, b in [(0,1), (0,2), (1,2)]:
        key = f"{names[a]}_{names[b]}"
        out[key + "_shared_centers"] = int(((split == a) & (split == b)).sum())
        out[key + "_shared_raw_pixels"] = int((footprint[a] & footprint[b]).sum())
        near = maximum_filter((split == a).astype(np.uint8), size=2*patch_size-1,
                              mode="constant", cval=0) > 0
        out[key + "_overlapping_centers_in_second_set"] = int((near & (split == b)).sum())
    return out

overlap = audit_overlap(split_dev, CAU_HINH["patch_size"])
overlap_df = pd.DataFrame(list(overlap.items()), columns=["metric", "count"])
display(overlap_df)

overlap_df.to_csv(RUN / "tables" / "development_overlap_audit.csv", index=False)
luu_json(RUN / "development" / "overlap_audit.json", overlap)

n_val = len(coords_dev_val)
n_test = len(coords_locked_test)
print("Validation patch overlap train:",
      f'{overlap["train_validation_overlapping_centers_in_second_set"]}/{n_val}',
      f'= {100*overlap["train_validation_overlapping_centers_in_second_set"]/n_val:.2f}%')
print("Test patch overlap train:",
      f'{overlap["train_test_overlapping_centers_in_second_set"]}/{n_test}',
      f'= {100*overlap["train_test_overlapping_centers_in_second_set"]/n_test:.2f}%')

In [ ]:
# ============================================================
# 5.3 TRỰC QUAN DEVELOPMENT SPLIT VÀ Ý NGHĨA OVERLAP
# ============================================================

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(split_dev + 1, cmap="viridis", vmin=0, vmax=3)
ax[0].set_title("Development train / validation / locked test")
ax[0].axis("off")

# Minh họa hình học bằng 2 patch giả định trên lưới 25x25.
canvas = np.zeros((25, 25), dtype=int)
train_c = (12, 10)
val_c = (12, 15)
r = CAU_HINH["patch_size"] // 2
canvas[train_c] = 1
canvas[val_c] = 2
ax[1].imshow(canvas, cmap="viridis", vmin=0, vmax=2)
rect1 = plt.Rectangle((train_c[1]-r-.5, train_c[0]-r-.5), 2*r+1, 2*r+1, fill=False, linewidth=2)
rect2 = plt.Rectangle((val_c[1]-r-.5, val_c[0]-r-.5), 2*r+1, 2*r+1, fill=False, linewidth=2)
ax[1].add_patch(rect1); ax[1].add_patch(rect2)
ax[1].scatter([train_c[1], val_c[1]], [train_c[0], val_c[0]], s=45)
ax[1].set_title("Tâm khác nhau nhưng patch 11×11 vẫn có thể giao nhau")
ax[1].set_xlim(2, 23); ax[1].set_ylim(23, 2)
fig.tight_layout()
luu_hinh(fig, "07_development_split_va_overlap")

**Kết quả này nói gì?** Center train/validation/test không trùng nhau, nhưng footprint của patch 11×11 vẫn có thể dùng chung raw pixels. Vì vậy validation chỉ dùng để **chọn cấu hình**, còn kết quả test cuối phải được mô tả đúng phạm vi là within-scene benchmark.

## 6. PCA + MinMax cho development

PCA và MinMax là phép biến đổi có tham số học từ dữ liệu. Để không dùng thống kê validation/test, development PCA/MinMax chỉ được fit trên **tâm development train**.

In [ ]:
DEV_PREP = RUN / 'development' / 'preprocessing_dev.joblib'
dev_prep = save_prep_checked(DEV_PREP, coords_dev_train)
pca_dev, scaler_dev = dev_prep['pca'], dev_prep['scaler']
flat = hsi.reshape(-1, hsi.shape[-1]).astype(np.float64)
processed_dev = scaler_dev.transform(pca_dev.transform(flat)).reshape(145,145,-1).astype(np.float32)
del flat
assert np.isfinite(processed_dev).all()
print('PCA development fit:', pca_dev.n_samples_, '| retained variance:', pca_dev.explained_variance_ratio_.sum())


In [ ]:
# ============================================================
# 6.2 TRỰC QUAN PCA
# ============================================================

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].plot(np.arange(1, CAU_HINH["pca_components"]+1),
           np.cumsum(pca_dev.explained_variance_ratio_) * 100)
ax[0].set_xlabel("Số thành phần PCA")
ax[0].set_ylabel("Phương sai tích lũy (%)")
ax[0].set_title("Thông tin được giữ sau PCA")
ax[0].grid(alpha=.2)

ax[1].imshow(processed_dev[...,0], cmap="viridis")
ax[1].set_title("Thành phần PCA 1")
ax[1].axis("off")
ax[2].imshow(processed_dev[...,1], cmap="viridis")
ax[2].set_title("Thành phần PCA 2")
ax[2].axis("off")
fig.tight_layout()
luu_hinh(fig, "08_pca_development")

**Kết quả này nói gì?** 200 dải phổ được nén còn 40 thành phần. Con số phần trăm phương sai được in từ dữ liệu thật, không ghi cứng. Validation/test chỉ được `transform`, không được fit lại.

In [ ]:
DEV_CACHE = RUN / 'development' / 'dev_train_val.npz'
DEV_CONTRACT = cache_contract(kind='dev_cache', train_coords=array_sha(coords_dev_train),
    val_coords=array_sha(coords_dev_val), preprocessing_sha256=sha256_file(DEV_PREP))
if not DEV_CACHE.exists():
    save_npz_checked(DEV_CACHE, DEV_CONTRACT,
        X_train=trich_patch(processed_dev, coords_dev_train, CAU_HINH['patch_size']),
        y_train=gt[coords_dev_train[:,0], coords_dev_train[:,1]].astype(np.int64)-1,
        coords_train=coords_dev_train,
        X_val=trich_patch(processed_dev, coords_dev_val, CAU_HINH['patch_size']),
        y_val=gt[coords_dev_val[:,0], coords_dev_val[:,1]].astype(np.int64)-1,
        coords_val=coords_dev_val)
verify_npz(DEV_CACHE, DEV_CONTRACT)
with np.load(DEV_CACHE, allow_pickle=False) as d:
    np.testing.assert_array_equal(d['coords_train'], coords_dev_train)
    np.testing.assert_array_equal(d['coords_val'], coords_dev_val)
    X_dev_train, y_dev_train, X_dev_val, y_dev_val = (d[k] for k in ['X_train','y_train','X_val','y_val'])
assert X_dev_train.shape == (4422, 11, 11, 40, 1)
assert X_dev_val.shape == (1106, 11, 11, 40, 1)
assert np.isfinite(X_dev_train).all() and np.isfinite(X_dev_val).all()
print('Development train / val:', X_dev_train.shape, X_dev_val.shape)
luu_json(RUN / 'cpu_preflight_pass.json', dict(run_id=run_id, passed=True,
    master_train=5528, test=4721, dev_train=4422, dev_val=1106,
    train_preprocessing_verified=True, no_model_training=True,
    validation_patch_overlap_expected=True))


In [ ]:
# ============================================================
# 6.4 PATCH 11×11×40: NHÌN 2D VÀ 3D
# ============================================================

sample_idx = 0
patch = X_dev_train[sample_idx, ..., 0]
label = int(y_dev_train[sample_idx] + 1)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for j, pc in enumerate([0, 4, 9, 19]):
    ax[j].imshow(patch[...,pc], cmap="viridis")
    ax[j].scatter([CAU_HINH["patch_size"]//2], [CAU_HINH["patch_size"]//2], s=30)
    ax[j].set_title(f"PC{pc+1} — lớp {label}")
    ax[j].axis("off")
fig.suptitle("Cùng một patch 11×11 ở nhiều thành phần PCA")
fig.tight_layout()
luu_hinh(fig, "09_patch_2d_nhieu_pca")

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
yy, xx = np.mgrid[0:11, 0:11]
for pc in [0, 5, 10, 20, 39]:
    face = plt.get_cmap("viridis")(stretch(patch[..., pc]))
    ax.plot_surface(xx, yy, np.full_like(xx, pc), facecolors=face, shade=False, alpha=.8)
ax.set_xlabel("Cột patch")
ax.set_ylabel("Hàng patch")
ax.set_zlabel("Thành phần PCA")
ax.set_title("Một patch HSI: 11×11×40")
ax.view_init(elev=24, azim=-55)
fig.tight_layout()
luu_hinh(fig, "10_patch_3d")

**Cách đọc patch:** nhãn của patch là nhãn ở pixel tâm. 11×11 biểu diễn ngữ cảnh không gian; 40 là số thành phần PCA. Đây là tensor mà mạng tích chập 3D nhận vào.

## 7. Khởi tạo TensorFlow/tf-keras và định nghĩa kiến trúc

Ba CNN baseline dùng cùng kiến trúc bộ phân lớp của 3D-HyperGAMO. Khác biệt chính giữa ba baseline là **cách xử lý mất cân bằng**.

Riêng HyperGAMO dùng MSE, Adam `β1=0.5` và chuỗi cập nhật đối kháng riêng. Vì vậy bảng 4 phương pháp không phải thí nghiệm chỉ thay đúng một yếu tố; notebook bổ sung `C-real-only matched` để giảm confound này.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # Runtime setup: never upgrade TensorFlow inside an already imported kernel.
    import importlib.metadata as im
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    try:
        tf_version = im.version('tensorflow')
    except im.PackageNotFoundError:
        try:
            tf_version = im.version('tensorflow-cpu')
        except im.PackageNotFoundError:
            raise RuntimeError('Môi trường thiếu TensorFlow. Dùng Kaggle GPU runtime hoặc cài TensorFlow trước rồi restart kernel.')
    tf_minor = '.'.join(tf_version.split('.')[:2])
    try:
        keras_version = im.version('tf-keras')
    except im.PackageNotFoundError:
        raise RuntimeError(f'Thiếu tf-keras. Chạy !pip install -q "tf-keras=={tf_minor}.*" rồi RESTART kernel và chạy lại.')
    if '.'.join(keras_version.split('.')[:2]) != tf_minor:
        raise RuntimeError(f'TensorFlow {tf_version} không khớp tf-keras {keras_version}. Cài tf-keras=={tf_minor}.* rồi restart.')
    import tensorflow as tf
    import tf_keras as keras
    devices = tf.config.list_physical_devices('GPU')
    if devices:
        index = int(CAU_HINH['gpu_index'])
        if not 0 <= index < len(devices):
            raise RuntimeError('gpu_index ngoài danh sách GPU')
        tf.config.set_visible_devices(devices[index], 'GPU')
        tf.config.experimental.set_memory_growth(devices[index], True)
    keras.utils.set_random_seed(CAU_HINH['development_seed'])
    tf.config.experimental.enable_op_determinism()
    luu_json(RUN / 'runtime_environment.json', dict(python=platform.python_version(),
        tensorflow=tf.__version__, tf_keras=keras.__version__, gpu=[d.name for d in devices],
        numpy=np.__version__, resume_caveat='Dropout/TF RNG after restart need not be bitwise identical'))
    print('TensorFlow / tf-keras:', tf.__version__, keras.__version__, '| GPU:', devices)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 7.2 BỘ PHÂN LỚP 3D-CNN DÙNG CHUNG
    # ============================================================

    def tao_classifier(shape, so_lop=16):
        L = keras.layers
        inp = L.Input(shape=shape)
        x = L.Conv3D(8, (1,1,7), activation="relu", padding="same")(inp)
        x = L.Conv3D(16, (3,3,5), activation="relu", padding="same")(x)
        x = L.Conv3D(32, (5,5,7), activation="relu", padding="same")(x)
        s = x.shape
        x = L.Reshape((s[1], s[2], s[3] * s[4]))(x)
        x = L.Conv2D(64, (3,3), activation="relu", padding="same")(x)
        x = L.GlobalAveragePooling2D()(x)
        x = L.Dropout(0.5)(x)
        out = L.Dense(so_lop, activation="softmax", kernel_initializer="he_normal")(x)
        return keras.Model(inp, out, name="classifier_3d_hypergamo")

    SHAPE = (CAU_HINH["patch_size"], CAU_HINH["patch_size"], CAU_HINH["pca_components"], 1)
    model_demo = tao_classifier(SHAPE, 16)
    model_demo.summary()
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 7.3 SƠ ĐỒ TRỰC QUAN KIẾN TRÚC CLASSIFIER
    # ============================================================

    labels = [
        "11×11×40×1\nInput",
        "Conv3D\n8 filters",
        "Conv3D\n16 filters",
        "Conv3D\n32 filters",
        "Reshape",
        "Conv2D\n64 filters",
        "Global Avg Pool",
        "Dropout 0.5",
        "Softmax\n16 lớp",
    ]
    fig, ax = plt.subplots(figsize=(16, 3))
    ax.axis("off")
    xs = np.linspace(.05, .95, len(labels))
    for i, (x, lab) in enumerate(zip(xs, labels)):
        ax.text(x, .5, lab, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.5", alpha=.15), transform=ax.transAxes)
        if i < len(labels)-1:
            ax.annotate("", xy=(xs[i+1]-.04,.5), xytext=(x+.04,.5),
                        xycoords=ax.transAxes, arrowprops=dict(arrowstyle="->"))
    ax.set_title("Kiến trúc bộ phân lớp dùng chung")
    fig.tight_layout()
    luu_hinh(fig, "11_so_do_classifier")
    del model_demo
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 8. Hàm tuning CNN — seed 42, validation chỉ dùng để chọn cấu hình

Vấn đề của notebook A là Macro-F1 validation có thể đạt 1.0 sớm và nhiều cấu hình bị hòa. B dùng **class-balanced validation NLL** làm score liên tục:

\[
L_{bal}=\frac{1}{C}\sum_c \frac{1}{n_c}\sum_{i:y_i=c}-\log p(y_i\mid x_i)
\]

Mỗi lớp đóng góp ngang nhau nên lớp lớn không lấn át lớp nhỏ.

Quy tắc chọn: **NLL cân bằng thấp hơn → nếu gần bằng thì Macro-F1 cao hơn → nếu vẫn hòa thì epoch sớm hơn.**
Balanced NLL là tiêu chí phát triển được định trước của notebook này, không phải
một cách xóa chồng lấn. Validation Oats chỉ có 2 mẫu nên lựa chọn vẫn có độ bất định.


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 8.1 ROS, tuning và checkpoint phục hồi
    def chi_so_ros(y, seed):
        """ROS giữ mọi mẫu gốc một lần rồi chỉ thêm các bản lặp còn thiếu."""
        y = np.asarray(y)
        rng = np.random.default_rng(seed)
        counts = np.bincount(y, minlength=16)
        if np.any(counts == 0):
            raise ValueError("ROS cần đủ 16 lớp trong train.")
        target = int(counts.max())
        parts = []
        for c in range(16):
            idx = np.flatnonzero(y == c)
            extra = rng.choice(idx, size=target-len(idx), replace=True)
            parts.append(np.concatenate([idx, extra]))
        out = np.concatenate(parts)
        rng.shuffle(out)
        assert np.array_equal(np.unique(out), np.arange(len(y)))
        assert np.all(np.bincount(y[out], minlength=16) == target)
        return out


    def class_weights_balanced(y):
        counts = np.bincount(y, minlength=16).astype(float)
        if np.any(counts == 0):
            raise ValueError("Class weight cần đủ 16 lớp trong train.")
        return {c: float(len(y)/(16*counts[c])) for c in range(16)}


    def _allow_training_mode(expected, label):
        if CHE_DO_CHAY != expected:
            print(f"Bỏ qua {label}: CHE_DO_CHAY={CHE_DO_CHAY}; cần {expected}.")
            return False
        require_gpu_training()
        return True


    def _finite_loss(value, label):
        values = np.asarray(value, dtype=float).reshape(-1)
        if not len(values) or not np.isfinite(values).all():
            raise FloatingPointError(f"Loss không hữu hạn: {label}: {values}")
        return float(values[0])


    def _initialize_optimizer(optimizer, variables):
        # Tạo toàn bộ slot trước restore; legacy Adam được dùng cho mã HyperGAMO.
        variables = list(variables)
        if hasattr(optimizer, "_create_all_weights"):
            optimizer._create_all_weights(variables)
        elif hasattr(optimizer, "build"):
            optimizer.build(variables)


    def _finite_model_weights(models):
        for model in models:
            for weight in model.weights:
                if not np.isfinite(weight.numpy()).all():
                    raise FloatingPointError(f"Weight không hữu hạn: {model.name}/{weight.name}")


    def _save_recovery(folder, objects, state):
        """Checkpoint optimizer + model, sau đó atomically công bố state tương ứng.

        Phục hồi có state NumPy và vị trí batch; không tuyên bố bitwise cho dropout
        TensorFlow hay kernel GPU. Giữ hai checkpoint để state cũ còn dùng được
        nếu phiên dừng giữa bước ghi checkpoint và bước công bố JSON.
        """
        folder = Path(folder)
        recovery = folder / "recovery"
        recovery.mkdir(parents=True, exist_ok=True)
        checkpoint = tf.train.Checkpoint(**objects)
        manager = tf.train.CheckpointManager(checkpoint, str(recovery/"tf"), max_to_keep=2)
        number = int(state.get("updates", state.get("epoch", 0)))
        prefix = manager.save(checkpoint_number=number)
        payload = dict(state)
        payload["runtime"] = {"tensorflow":tf.__version__,"tf_keras":keras.__version__}
        payload["checkpoint"] = Path(prefix).name
        payload["resume_policy"] = "optimizer/model/NumPy/batch state; not bitwise TF dropout"
        luu_json(recovery / "state.json", payload)


    def _restore_recovery(folder, objects, contract):
        recovery = Path(folder) / "recovery"
        state_file = recovery / "state.json"
        if not state_file.exists():
            return None
        state = doc_json(state_file)
        if state.get("runtime") != {"tensorflow":tf.__version__,"tf_keras":keras.__version__}:
            raise RuntimeError("Recovery cần đúng TensorFlow/tf-keras đã tạo checkpoint.")
        if state.get("contract") != contract:
            raise RuntimeError(f"Recovery contract khác cấu hình hiện tại: {folder}")
        prefix = recovery / "tf" / state["checkpoint"]
        if not Path(str(prefix)+".index").exists():
            raise RuntimeError(f"Checkpoint thiếu: {prefix}")
        checkpoint = tf.train.Checkpoint(**objects)
        # Tạo save_counter để mọi object sẵn sàng được đối chiếu khi restore.
        _ = checkpoint.save_counter
        checkpoint.restore(str(prefix)).assert_existing_objects_matched()
        print("Phục hồi", folder, "tại", state.get("updates", state.get("epoch")))
        return state


    class TheoDoiValidation(keras.callbacks.Callback):
        """Balanced NLL là tiêu chí chính; checkpoint mỗi epoch để phục hồi."""
        def __init__(self, X_val, y_val, patience, min_delta, batch_size,
                     folder, contract, restored=None):
            super().__init__()
            self.X_val, self.y_val = X_val, y_val
            self.patience, self.min_delta = patience, min_delta
            self.batch_size, self.folder, self.contract = batch_size, Path(folder), contract
            restored = restored or {}
            self.rows = restored.get("rows", [])
            key = restored.get("best_key")
            self.best_key = tuple(key) if key is not None else None
            self.wait = int(restored.get("wait", 0))
            self.best_weights_path = self.folder / restored["best_weights"] if restored.get("best_weights") else None

        def on_batch_end(self, batch, logs=None):
            _finite_loss((logs or {}).get("loss", np.nan), "CNN tuning batch")

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            prob = self.model.predict(self.X_val, batch_size=self.batch_size, verbose=0)
            if not np.isfinite(prob).all():
                raise FloatingPointError("Validation probabilities không hữu hạn.")
            pred = prob.argmax(1)
            bnll = balanced_nll(self.y_val, prob)
            _finite_loss(bnll, "balanced NLL")
            mf1 = float(f1_score(self.y_val, pred, average="macro", labels=np.arange(16), zero_division=0))
            oa = float(accuracy_score(self.y_val, pred))
            row = {
                "epoch": int(epoch+1),
                "optimizer_updates": int(self.model.optimizer.iterations.numpy()),
                "loss": _finite_loss(logs.get("loss", np.nan), "CNN tuning epoch"),
                "val_balanced_nll": float(bnll), "val_macro_f1": mf1, "val_oa": oa,
            }
            self.rows.append(row)
            key = (float(bnll), -mf1, int(epoch+1))
            if self.best_key is None or bnll < self.best_key[0]-self.min_delta or (
                abs(bnll-self.best_key[0]) <= self.min_delta and key[1:] < self.best_key[1:]
            ):
                self.best_key, self.wait = key, 0
                self.best_weights_path = self.folder / f"best_epoch_{epoch+1:03d}.weights.h5"
                self.model.save_weights(self.best_weights_path)
            else:
                self.wait += 1
                if self.wait >= self.patience:
                    self.model.stop_training = True
            pd.DataFrame(self.rows).to_csv(self.folder/"history.csv", index=False)
            _save_recovery(self.folder, {"model": self.model, "optimizer": self.model.optimizer}, {
                "contract": self.contract, "epoch": int(epoch+1),
                "updates": int(self.model.optimizer.iterations.numpy()),
                "rows": self.rows, "best_key": list(self.best_key), "wait": self.wait,
                "best_weights": self.best_weights_path.name,
            })
            # Chỉ xóa best cũ SAU KHI state mới đã được công bố thành công.
            for obsolete in self.folder.glob("best_epoch_*.weights.h5"):
                if obsolete != self.best_weights_path:
                    obsolete.unlink()


    def du_lieu_cnn(method, X, y, seed):
        if method == "cnn_thuong":
            return X, y, None
        if method == "ros":
            idx = chi_so_ros(y, seed)
            return X[idx], y[idx], None
        if method == "class_weight":
            return X, y, class_weights_balanced(y)
        raise ValueError(method)


    def tuning_cnn(method, display_name):
        folder = RUN / "development" / "tuning" / method
        folder.mkdir(parents=True, exist_ok=True)
        if load_done_checked(folder, ["best.json"]) is not None:
            return doc_json(folder/"best.json")
        if not _allow_training_mode("development", "tuning "+display_name):
            return None
        trials = []
        for lr in CAU_HINH["cnn_learning_rates"]:
            trial_folder = folder / f"lr_{lr:.0e}"
            trial_folder.mkdir(parents=True, exist_ok=True)
            completed = load_done_checked(trial_folder, ["best.weights.h5", "trial.json"])
            if completed is not None:
                trials.append(doc_json(trial_folder/"trial.json"))
                continue
            seed = CAU_HINH["development_seed"]
            keras.backend.clear_session()
            keras.utils.set_random_seed(seed)
            np.random.seed(seed)
            X_m, y_m, cw = du_lieu_cnn(method, X_dev_train, y_dev_train, seed=4200)
            model = tao_classifier(SHAPE, 16)
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                          loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
            _initialize_optimizer(model.optimizer, model.trainable_variables)
            contract = {"method": method, "seed": int(seed), "lr": float(lr),
                        "batch_size": int(CAU_HINH["cnn_batch_size"]),
                        "presentations": int(len(y_m)),
                        "max_epochs": int(CAU_HINH["cnn_tuning_max_epochs"]),
                        "patience": int(CAU_HINH["cnn_tuning_patience"]),
                        "min_delta": float(CAU_HINH["cnn_min_delta_nll"])}
            restored = _restore_recovery(trial_folder, {"model": model, "optimizer": model.optimizer}, contract)
            tracker = TheoDoiValidation(X_dev_val, y_dev_val,
                CAU_HINH["cnn_tuning_patience"], CAU_HINH["cnn_min_delta_nll"],
                CAU_HINH["eval_batch_size"], trial_folder, contract, restored)
            initial_epoch = int((restored or {}).get("epoch", 0))
            if initial_epoch < contract["max_epochs"] and tracker.wait < contract["patience"]:
                print(f"\n--- {display_name}: LR={lr}; bắt đầu epoch {initial_epoch+1} ---")
                model.fit(X_m, y_m, batch_size=contract["batch_size"],
                    initial_epoch=initial_epoch, epochs=contract["max_epochs"], shuffle=True,
                    class_weight=cw, callbacks=[tracker], verbose=2)
            if tracker.best_key is None or tracker.best_weights_path is None or not tracker.best_weights_path.exists():
                raise RuntimeError("Không có checkpoint tuning hợp lệ.")
            model.load_weights(tracker.best_weights_path)
            _finite_model_weights([model])
            model.save_weights(trial_folder/"best.weights.h5")
            hist = pd.DataFrame(tracker.rows)
            hist.to_csv(trial_folder/"history.csv", index=False)
            row = hist.loc[hist.epoch == int(tracker.best_key[2])].iloc[0]
            trial = {"learning_rate": float(lr), "best_epoch": int(row.epoch),
                "fixed_optimizer_updates": int(row.optimizer_updates),
                "development_steps_per_epoch": int(math.ceil(len(y_m)/contract["batch_size"])),
                "development_presentations": int(len(y_m)),
                "val_balanced_nll": float(row.val_balanced_nll),
                "val_macro_f1": float(row.val_macro_f1), "val_oa": float(row.val_oa),
                "history": str((trial_folder/"history.csv").relative_to(RUN)),
                "weights": str((trial_folder/"best.weights.h5").relative_to(RUN))}
            luu_json(trial_folder/"trial.json", trial)
            stamp_done(trial_folder, {"complete": True, "method": method,
                "seed": int(seed), "learning_rate": float(lr)},
                [trial_folder/"best.weights.h5", trial_folder/"trial.json"])
            trials.append(trial)
            luu_json(folder/"trials_so_far.json", trials)
        best = sorted(trials, key=lambda t: (t["val_balanced_nll"], -t["val_macro_f1"],
                                           t["best_epoch"], t["learning_rate"]))[0]
        result = {"method": method, "display_name": display_name,
            "seed": int(CAU_HINH["development_seed"]),
            "selection_rule": "min balanced NLL; tie -> max Macro-F1; tie -> earlier epoch",
            "final_budget_rule": "replay selected optimizer update count at constant LR",
            "best": best, "trials": trials}
        luu_json(folder/"best.json", result)
        stamp_done(folder, {"complete": True, "method": method, "stage": "development"},
                   [folder/"best.json"]+[RUN/t["weights"] for t in trials])
        return result


    def ve_tuning(result):
        if result is None:
            return
        trials = pd.DataFrame(result["trials"])
        display(trials[["learning_rate", "best_epoch", "fixed_optimizer_updates",
                        "val_balanced_nll", "val_macro_f1", "val_oa"]])
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
        for t in result["trials"]:
            h = pd.read_csv(RUN/t["history"])
            ax[0].plot(h.optimizer_updates, h.val_balanced_nll, label=f'LR={t["learning_rate"]:g}')
            ax[1].plot(h.optimizer_updates, h.val_macro_f1, label=f'LR={t["learning_rate"]:g}')
        ax[0].set_title("Validation NLL cân bằng")
        ax[1].set_title("Validation Macro-F1")
        for a in ax:
            a.set_xlabel("Optimizer updates"); a.grid(alpha=.2); a.legend(fontsize=8)
        fig.suptitle(result["display_name"]); fig.tight_layout()
        luu_hinh(fig, f'12_tuning_{result["method"]}')
        print(json.dumps(result["best"], ensure_ascii=False, indent=2))

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 9. Tuning CNN thường

CNN thường **không xử lý mất cân bằng**. Đây là mốc nền để biết bản thân kiến trúc classifier làm được gì trước khi ROS/Class Weight/GAN can thiệp.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # TRAIN/TUNING RIÊNG — CNN THƯỜNG
    ket_qua_tuning_cnn = tuning_cnn("cnn_thuong", "CNN thường")
    ve_tuning(ket_qua_tuning_cnn)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


**Kết quả cần đọc:** NLL cân bằng cho biết mức tự tin theo lớp; Macro-F1 cho biết chất lượng phân loại cân bằng giữa 16 lớp. `best_epoch` và `learning_rate` được lưu vào file, các seed final sau này chỉ dùng lại cấu hình này, không tự săn checkpoint mới.

## 10. Tuning CNN + Random Oversampling (ROS)

ROS không tạo thông tin mới. Nó **lặp lại** mẫu train của lớp ít cho tới khi số lần xuất hiện của các lớp ngang nhau. Validation và test không bị oversample.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 10.1 Trực quan trước/sau ROS trên development train
    before = np.bincount(y_dev_train, minlength=16)
    idx_demo_ros = chi_so_ros(y_dev_train, seed=4200)
    after = np.bincount(y_dev_train[idx_demo_ros], minlength=16)
    fig, ax = plt.subplots(1,2,figsize=(14,4.5))
    ax[0].bar(ID_LOP, before); ax[0].set_title("Trước ROS")
    ax[1].bar(ID_LOP, after); ax[1].set_title("Sau ROS — số lần xuất hiện")
    for a in ax:
        a.set_xticks(ID_LOP); a.set_xlabel("ID lớp"); a.set_ylabel("Số mẫu/lượt")
    fig.tight_layout()
    luu_hinh(fig, "13_ros_truoc_sau")
    print("ROS không tạo pixel/patch mới; chỉ lặp lại chỉ số mẫu train.")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # TRAIN/TUNING RIÊNG — CNN + ROS
    ket_qua_tuning_ros = tuning_cnn("ros", "CNN + Random Oversampling")
    ve_tuning(ket_qua_tuning_ros)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 11. Tuning CNN + Class Weight

Class Weight không lặp mẫu. Thay vào đó, lỗi của lớp hiếm được nhân trọng số lớn hơn trong hàm mất mát.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 11.1 Trực quan class weight
    cw_demo = class_weights_balanced(y_dev_train)
    weights_arr = np.array([cw_demo[i] for i in range(16)])
    fig, ax = plt.subplots(1,2,figsize=(14,4.5))
    ax[0].bar(ID_LOP, np.bincount(y_dev_train, minlength=16))
    ax[0].set_title("Số mẫu development train")
    ax[1].bar(ID_LOP, weights_arr)
    ax[1].set_title("Trọng số dùng trong loss")
    for a in ax:
        a.set_xticks(ID_LOP); a.set_xlabel("ID lớp"); a.grid(axis="y", alpha=.2)
    fig.tight_layout()
    luu_hinh(fig, "14_class_weight")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # TRAIN/TUNING RIÊNG — CNN + CLASS WEIGHT
    ket_qua_tuning_cw = tuning_cnn("class_weight", "CNN + Class Weight")
    ve_tuning(ket_qua_tuning_cw)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 12. Khóa cấu hình CNN sau development

Từ thời điểm này, CNN final không được nhìn validation để chọn checkpoint riêng theo seed. Mỗi phương pháp dùng **một LR + một số epoch đã chọn từ seed 42**.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 12.1 Khóa learning rate và CHÍNH XÁC số optimizer updates
    _tuning_results = {
        "cnn_thuong": ket_qua_tuning_cnn,
        "ros": ket_qua_tuning_ros,
        "class_weight": ket_qua_tuning_cw,
    }
    cnn_frozen = {k: v["best"] for k, v in _tuning_results.items() if v is not None}
    if len(cnn_frozen) == 3:
        for method, cfg in cnn_frozen.items():
            if int(cfg["fixed_optimizer_updates"]) <= 0:
                raise RuntimeError("Ngân sách CNN phải dương.")
        immutable_json(RUN/"cnn_frozen_config.json", cnn_frozen)
        frozen_table = pd.DataFrame([{
            "method": k, "learning_rate": v["learning_rate"],
            "selected_development_epoch": v["best_epoch"],
            "fixed_optimizer_updates": v["fixed_optimizer_updates"],
            "dev_balanced_nll": v["val_balanced_nll"],
            "dev_macro_f1": v["val_macro_f1"],
        } for k, v in cnn_frozen.items()])
        display(frozen_table)
        frozen_table.to_csv(RUN/"tables"/"cnn_frozen_config.csv", index=False)
    else:
        print("Chưa đủ tuning CNN. Chế độ preflight không chạy tuning/final training.")
        if CHE_DO_CHAY == "baseline":
            raise RuntimeError("Cần hoàn tất chế độ development trước baseline.")

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


**Kết quả này nói gì?** Development đã hoàn thành vai trò của nó. Khi chuyển sang final training, toàn bộ 5528 master train được sử dụng; không giữ lại 20% validation nữa.

## 13. Chuẩn bị dữ liệu FINAL — PCA/MinMax fit lại trên toàn bộ 5528 train

Đây là dữ liệu dùng cho mọi final model. Test được transform rồi lưu vào file khóa; các hàm training phía sau **không đọc file test**.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    FINAL_PREP = RUN / 'final_data' / 'preprocessing_final.joblib'
    final_prep = save_prep_checked(FINAL_PREP, coords_master_train)
    pca_final, scaler_final = final_prep['pca'], final_prep['scaler']
    flat = hsi.reshape(-1, hsi.shape[-1]).astype(np.float64)
    processed_final = scaler_final.transform(pca_final.transform(flat)).reshape(145,145,-1).astype(np.float32)
    del flat
    assert np.isfinite(processed_final).all()
    FINAL_TRAIN_CACHE = RUN / 'final_data' / 'train_5528.npz'
    LOCKED_TEST_CACHE = RUN / 'final_data' / 'test_4721_LOCKED.npz'
    FINAL_CONTRACT = cache_contract(kind='final_train', coords=array_sha(coords_master_train),
        preprocessing_sha256=sha256_file(FINAL_PREP))
    if not FINAL_TRAIN_CACHE.exists():
        save_npz_checked(FINAL_TRAIN_CACHE, FINAL_CONTRACT,
            X_train=trich_patch(processed_final, coords_master_train, CAU_HINH['patch_size']),
            y_train=gt[coords_master_train[:,0], coords_master_train[:,1]].astype(np.int64)-1,
            coords_train=coords_master_train)
    verify_npz(FINAL_TRAIN_CACHE, FINAL_CONTRACT)
    with np.load(FINAL_TRAIN_CACHE, allow_pickle=False) as d:
        X_final_train, y_final_train = d['X_train'], d['y_train']
        np.testing.assert_array_equal(d['coords_train'], coords_master_train)
    assert X_final_train.shape == (5528, 11, 11, 40, 1)
    assert np.isfinite(X_final_train).all()
    # Test patches are only materialized after the final test registry has been frozen.
    print('Final train:', X_final_train.shape, '| PCA fit:', pca_final.n_samples_)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 14. Final CNN — ngân sách cập nhật đã khóa

Final train khởi tạo từ đầu trên 5528 mẫu, LR cố định; giữ **đúng số optimizer updates**
đã chọn ở development (kể cả dừng giữa epoch). Không giữ cùng epoch rồi vô tình tăng
ngân sách khoảng 25%. ROS giữ mọi mẫu thật và chỉ thêm bản lặp để cân bằng lớp.


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 14.1 Final CNN: constant LR, đúng ngân sách updates, phục hồi mỗi epoch
    def train_final_cnn(method, display_name, seed):
        folder = RUN/"models"/method/f"seed_{seed}"
        folder.mkdir(parents=True, exist_ok=True)
        completed = load_done_checked(folder, ["classifier.weights.h5"])
        if completed is not None:
            return completed
        if not _allow_training_mode("baseline", "final "+display_name):
            return None
        if method not in cnn_frozen:
            raise RuntimeError("Thiếu cấu hình CNN đã khóa từ development.")
        cfg = cnn_frozen[method]
        lr, target = float(cfg["learning_rate"]), int(cfg["fixed_optimizer_updates"])
        if target <= 0:
            raise ValueError("fixed_optimizer_updates phải dương.")
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        np.random.seed(seed)
        rng = np.random.default_rng(seed)
        X_m, y_m, cw = du_lieu_cnn(method, X_final_train, y_final_train, seed)
        model = tao_classifier(SHAPE, 16)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
        _initialize_optimizer(model.optimizer, model.trainable_variables)
        bs, n = int(CAU_HINH["cnn_batch_size"]), int(len(y_m))
        contract = {"method": method, "seed": int(seed), "lr": lr,
            "target_updates": target, "batch_size": bs, "presentations": n}
        objects = {"model": model, "optimizer": model.optimizer}
        restored = _restore_recovery(folder, objects, contract)
        if restored:
            rng.bit_generator.state = restored["rng_state"]
            updates, epoch, ptr = int(restored["updates"]), int(restored["epoch"]), int(restored["ptr"])
            indices = np.asarray(restored["indices"], dtype=np.int64)
            rows, accum = restored["rows"], np.asarray(restored["accum"], dtype=float)
            elapsed_base = float(restored["elapsed_minutes"])
        else:
            updates, epoch, ptr = 0, 1, 0
            indices = rng.permutation(n)
            rows, accum, elapsed_base = [], np.zeros(3), 0.0
        if int(model.optimizer.iterations.numpy()) != updates:
            raise RuntimeError("Optimizer step và recovery step không khớp.")
        t0 = time.time()
        print(f"FINAL {display_name}; seed={seed}; constant lr={lr}; updates={updates}/{target}")
        while updates < target:
            if ptr == n:
                indices, ptr, epoch = rng.permutation(n), 0, epoch+1
            end = min(ptr+bs, n)
            idx = indices[ptr:end]
            result = np.asarray(model.train_on_batch(X_m[idx], y_m[idx], class_weight=cw), dtype=float).reshape(-1)
            loss = _finite_loss(result, "final CNN")
            accuracy = float(result[1]) if len(result) > 1 else 0.0
            accum += [loss*len(idx), accuracy*len(idx), len(idx)]
            ptr, updates = end, updates+1
            if int(model.optimizer.iterations.numpy()) != updates:
                raise RuntimeError("Final CNN optimizer counter không khớp.")
            if ptr == n or updates == target:
                rows.append({"epoch": epoch, "optimizer_updates": updates,
                    "loss": float(accum[0]/accum[2]), "accuracy": float(accum[1]/accum[2]),
                    "epoch_complete": bool(ptr == n), "presentations_this_epoch": int(accum[2])})
                accum[:] = 0
                elapsed = elapsed_base+(time.time()-t0)/60
                pd.DataFrame(rows).to_csv(folder/"history.csv", index=False)
                _save_recovery(folder, objects, {"contract": contract, "updates": updates,
                    "epoch": epoch, "ptr": ptr, "indices": indices.tolist(),
                    "rng_state": rng.bit_generator.state, "rows": rows,
                    "accum": accum.tolist(), "elapsed_minutes": elapsed})
                print("CNN checkpoint", display_name, "seed", seed, "updates", updates, "/", target)
        _finite_model_weights([model])
        model.save_weights(folder/"classifier.weights.h5")
        meta = {"complete": True, "method": method, "display_name": display_name, "seed": int(seed),
            "learning_rate": lr, "fixed_optimizer_updates": target,
            "selected_development_epoch": int(cfg["best_epoch"]),
            "final_full_epochs": int(target//math.ceil(n/bs)),
            "batch_size": bs, "train_samples_or_presentations": n,
            "elapsed_minutes": elapsed_base+(time.time()-t0)/60,
            "budget_rule": "exact selected optimizer update count; constant LR; partial final epoch allowed",
            "resume_policy": "optimizer/model/NumPy/batch state; not bitwise TF dropout"}
        stamp_done(folder, meta, [folder/"classifier.weights.h5"])
        return meta


    def ve_history_final_cnn(method, display_name):
        fig, ax = plt.subplots(figsize=(9, 4.5))
        any_data = False
        for seed in CAU_HINH["seeds"]:
            p = RUN/"models"/method/f"seed_{seed}"/"history.csv"
            if p.exists():
                h = pd.read_csv(p)
                ax.plot(h.optimizer_updates, h.loss, label=f"seed {seed}")
                any_data = True
        if any_data:
            ax.set_xlabel("Optimizer updates"); ax.set_ylabel("Training loss")
            ax.set_title(display_name+" — final training"); ax.grid(alpha=.2); ax.legend()
            fig.tight_layout(); luu_hinh(fig, f"15_final_{method}_loss")
        else:
            plt.close(fig)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 15. FINAL — CNN thường

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # TRAIN RIÊNG: CNN THƯỜNG
    for seed in CAU_HINH["seeds"]:
        train_final_cnn("cnn_thuong", "CNN thường", seed)
    ve_history_final_cnn("cnn_thuong", "CNN thường")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 16. FINAL — CNN + Random Oversampling

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # TRAIN RIÊNG: CNN + ROS
    for seed in CAU_HINH["seeds"]:
        train_final_cnn("ros", "CNN + Random Oversampling", seed)
    ve_history_final_cnn("ros", "CNN + Random Oversampling")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 17. FINAL — CNN + Class Weight

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # TRAIN RIÊNG: CNN + CLASS WEIGHT
    for seed in CAU_HINH["seeds"]:
        train_final_cnn("class_weight", "CNN + Class Weight", seed)
    ve_history_final_cnn("class_weight", "CNN + Class Weight")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 18. Ablation — C-real-only matched về loss, budget và Adam clock

Giữ kiến trúc C, MSE, LR, beta1, 8000 lần cập nhật C, thứ tự real batches và batch cuối.
Mỗi vòng thay fake bằng một lần học real; Adam clock tiến qua các bước D/G bị bỏ qua,
giữ lịch bias correction của shared optimizer. Đây là đối chứng cơ chế; dropout/RNG
không được coi là cùng một đường ngẫu nhiên với GAN và không chứng minh nhân quả ngoài benchmark này.


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 18.1 C-real-only: cùng MSE, C budget, batch partition và optimizer clock
    def sorted_class_mapping(y):
        counts = np.bincount(y, minlength=16)
        if np.any(counts == 0):
            raise ValueError("Cần đủ 16 lớp.")
        order = np.argsort(counts, kind="mergesort")
        inverse = np.empty_like(order)
        inverse[order] = np.arange(16)
        return counts, order, inverse


    def train_matched_real_only(seed):
        method = "c_real_only_matched"
        folder = RUN/"models"/method/f"seed_{seed}"
        folder.mkdir(parents=True, exist_ok=True)
        completed = load_done_checked(folder, ["classifier.weights.h5", "class_order.json"])
        if completed is not None:
            return completed
        if not _allow_training_mode("baseline", "C-real-only clock control"):
            return None
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        rng = np.random.default_rng(seed)
        counts, order, inverse = sorted_class_mapping(y_final_train)
        Y = keras.utils.to_categorical(inverse[y_final_train], 16).astype(np.float32)
        C = tao_classifier(SHAPE, 16)
        opt = keras.optimizers.legacy.Adam(learning_rate=CAU_HINH["gan_learning_rate"],
                                          beta_1=CAU_HINH["gan_beta1"])
        C.compile(loss="mean_squared_error", optimizer=opt)
        _initialize_optimizer(opt, C.trainable_variables)
        n, bs = len(X_final_train), int(CAU_HINH["gan_batch_size"])
        target, n_heads = int(CAU_HINH["gan_classifier_updates"]), 15
        if target <= 0 or target % 2:
            raise ValueError("GAN/C-control budget phải là số chẵn dương.")
        n_batches = int(math.ceil(n/bs))
        clock_per_loop = 4+2*n_heads
        contract = {"method": method, "seed": int(seed), "target_updates": target,
            "lr": float(CAU_HINH["gan_learning_rate"]), "beta1": float(CAU_HINH["gan_beta1"]),
            "batch_size": bs, "samples": int(n), "clock_per_loop": clock_per_loop}
        objects = {"C": C, "optimizer": opt}
        restored = _restore_recovery(folder, objects, contract)
        if restored:
            rng.bit_generator.state = restored["rng_state"]
            real_order = np.asarray(restored["real_order"], dtype=np.int64)
            updates, loop = int(restored["updates"]), int(restored["loop"])
            rows, recent, recent_n = restored["rows"], float(restored["recent"]), int(restored["recent_n"])
            elapsed_base = float(restored["elapsed_minutes"])
        else:
            real_order = rng.permutation(n)
            updates, loop, rows, recent, recent_n, elapsed_base = 0, 0, [], 0.0, 0, 0.0
        if int(opt.iterations.numpy()) != loop*clock_per_loop:
            raise RuntimeError("C control optimizer clock không khớp recovery.")
        log_every = int(CAU_HINH["gan_log_every"])
        save_every = int(CAU_HINH.get("gan_checkpoint_every_updates", log_every))
        if log_every <= 0 or log_every % 2 or save_every <= 0 or save_every % 2:
            raise ValueError("GAN log/checkpoint cadence phải là số chẵn dương.")
        t0 = time.time()
        while updates < target:
            j = loop % n_batches
            idx = real_order[j*bs:min((j+1)*bs, n)]
            loss1 = _finite_loss(C.train_on_batch(X_final_train[idx], Y[idx]), "C real")
            opt.iterations.assign_add(1)  # D-real không tồn tại trong control.
            # Thay lượt C-fake bằng cùng real batch; empirical class prior giữ nguyên.
            loss2 = _finite_loss(C.train_on_batch(X_final_train[idx], Y[idx]), "C real replacement")
            opt.iterations.assign_add(1+2*n_heads)  # bỏ D-fake + hai lượt G cho mỗi head.
            updates, loop = updates+2, loop+1
            if int(opt.iterations.numpy()) != loop*clock_per_loop:
                raise RuntimeError("C control Adam clock không khớp HyperGAMO.")
            recent, recent_n = recent+(loss1+loss2)/2, recent_n+1
            elapsed = elapsed_base+(time.time()-t0)/60
            if updates % log_every == 0 or updates == target:
                rows.append({"update": updates, "optimizer_clock": int(opt.iterations.numpy()),
                    "mse_loss": recent/recent_n, "elapsed_minutes": elapsed})
                recent, recent_n = 0.0, 0
                pd.DataFrame(rows).to_csv(folder/"history.csv", index=False)
                print("C-real-only clock control", seed, updates, rows[-1]["mse_loss"])
            if updates % save_every == 0 or updates == target:
                _save_recovery(folder, objects, {"contract": contract, "updates": updates,
                    "loop": loop, "real_order": real_order.tolist(), "rng_state": rng.bit_generator.state,
                    "rows": rows, "recent": recent, "recent_n": recent_n, "elapsed_minutes": elapsed})
        _finite_model_weights([C])
        C.save_weights(folder/"classifier.weights.h5")
        luu_json(folder/"class_order.json", {"original_class_ids": (order+1).tolist()})
        meta = {"complete": True, "method": method, "seed": int(seed),
            "classifier_updates": target, "optimizer_clock": int(opt.iterations.numpy()),
            "learning_rate": float(CAU_HINH["gan_learning_rate"]), "beta1": float(CAU_HINH["gan_beta1"]),
            "elapsed_minutes": elapsed_base+(time.time()-t0)/60,
            "control_rule": "same C init/MSE/batch partition/update budget/shared-Adam C clock; fake slots replaced by real",
            "remaining_confound": "replacement class prior is empirical rather than GAN deficit-sampled",
            "resume_policy": "optimizer/model/NumPy/batch state; not bitwise TF dropout"}
        stamp_done(folder, meta, [folder/"classifier.weights.h5", folder/"class_order.json"])
        return meta


    for seed in CAU_HINH["seeds"]:
        train_matched_real_only(seed)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    _has_matched_history = False
    for seed in CAU_HINH["seeds"]:
        p = RUN/"models"/"c_real_only_matched"/f"seed_{seed}"/"history.csv"
        if p.exists():
            h = pd.read_csv(p); ax.plot(h["update"], h.mse_loss, label=f"seed {seed}")
            _has_matched_history = True
    if _has_matched_history:
        ax.set_title("C-real-only — MSE, budget và Adam clock control")
        ax.set_xlabel("Classifier updates"); ax.set_ylabel("MSE"); ax.grid(alpha=.2); ax.legend()
        fig.tight_layout(); luu_hinh(fig, "16_c_real_only_matched_loss")
    else:
        plt.close(fig)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 19. 3D-HyperGAMO — cơ chế sinh mẫu

HyperGAMO không sinh patch hoàn toàn tự do. Generator tạo một vector đặc trưng; mỗi lớp thiểu số có một `head` dùng `softmax` để sinh trọng số trên các mẫu thật cùng lớp:

\[
\tilde{x}=\sum_i w_i x_i,\qquad w_i\ge0,\quad \sum_i w_i=1
\]

Vì vậy mẫu sinh là **tổ hợp lồi** của các patch thật trong cùng lớp. Nếu một trọng số tiến sát 1, mẫu sinh có thể gần như sao chép một support mẫu thật — đây là lý do cần chẩn đoán `max_weight`, entropy và nearest-real distance.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 19.1 SƠ ĐỒ TRỰC QUAN HYPERGAMO
    # ============================================================

    fig, ax = plt.subplots(figsize=(15,5))
    ax.axis("off")
    items = [
        (.08,.62,"Noise z\n+ class"),
        (.28,.62,"Generator G\n→ vector 784"),
        (.50,.62,"Softmax head\n[w₁…wₙ]"),
        (.72,.62,"Σ wᵢ·realᵢ\nsynthetic patch"),
        (.90,.78,"Classifier C"),
        (.90,.45,"Discriminator D"),
    ]
    for x,y,t in items:
        ax.text(x,y,t,ha="center",va="center",transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=.5", alpha=.15))
    for a,b in [((.13,.62),(.23,.62)),((.34,.62),(.44,.62)),((.56,.62),(.66,.62)),((.78,.66),(.85,.76)),((.78,.58),(.85,.47))]:
        ax.annotate("",xy=b,xytext=a,xycoords=ax.transAxes,arrowprops=dict(arrowstyle="->"))
    ax.text(.50,.28,"Điểm then chốt: head chỉ trộn các mẫu thật cùng lớp\n→ dễ kiểm tra mức tập trung trọng số và near-copy",
            ha="center",va="center",transform=ax.transAxes)
    ax.set_title("Cơ chế 3D-HyperGAMO dùng trong notebook")
    fig.tight_layout()
    luu_hinh(fig, "17_so_do_hypergamo")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 19.2 ĐỊNH NGHĨA G, D VÀ HEAD TỔ HỢP LỒI
    # ============================================================

    def tao_generator(latent_dim, so_lop):
        L = keras.layers
        noise = L.Input(shape=(latent_dim,))
        labels = L.Input(shape=(so_lop,))
        x = L.Concatenate()([noise, labels])
        x = L.Dense(7*7*256, use_bias=False)(x)
        x = L.BatchNormalization(momentum=0.9)(x)
        x = L.LeakyReLU()(x)
        x = L.Reshape((7,7,256))(x)
        x = L.Conv2DTranspose(128, (5,5), strides=(1,1), padding="same", use_bias=False)(x)
        x = L.BatchNormalization(momentum=0.9)(x)
        x = L.LeakyReLU()(x)
        x = L.Conv2DTranspose(64, (5,5), strides=(2,2), padding="same", use_bias=False)(x)
        x = L.BatchNormalization(momentum=0.9)(x)
        x = L.LeakyReLU()(x)
        x = L.Conv2DTranspose(1, (5,5), strides=(2,2), padding="same", use_bias=False)(x)
        x = L.BatchNormalization(momentum=0.9)(x)
        out = L.Flatten()(x)  # 28×28 = 784
        return keras.Model([noise, labels], out, name="G")


    def tao_head_convex(support, shape):
        """Tương đương denseGenProcessCreate: softmax weights × support matrix."""
        L = keras.layers
        support_flat = support.reshape(len(support), -1).astype(np.float32)
        support_const = tf.constant(support_flat)
        inp = L.Input(shape=(784,))
        weights = L.Dense(len(support), activation="softmax", name="support_weights")(inp)
        flat = L.Lambda(lambda w: tf.matmul(w, support_const), name="convex_sum")(weights)
        out = L.Reshape(shape)(flat)
        return keras.Model(inp, out, name=f"head_{len(support)}")


    def tao_discriminator(shape, so_lop):
        L = keras.layers
        inp = L.Input(shape=shape)
        x = L.Conv3D(8,(1,1,7),activation="relu",padding="same")(inp)
        x = L.Conv3D(16,(3,3,5),activation="relu",padding="same")(x)
        x = L.Conv3D(32,(5,5,7),activation="relu",padding="same")(x)
        s = x.shape
        x = L.Reshape((s[1],s[2],s[3]*s[4]))(x)
        x = L.Conv2D(64,(3,3),activation="relu",padding="same")(x)
        x = L.GlobalAveragePooling2D()(x)
        label = L.Input(shape=(so_lop,))
        x = L.Concatenate()([x,label])
        x = L.Dropout(.5)(x)
        out = L.Dense(1,activation="sigmoid",kernel_initializer="he_normal")(x)
        return keras.Model([inp,label],out,name="D")


    def build_hypergamo_models(X, y_original, seed):
        """Dựng toàn bộ C/D/G/heads. Supports lấy theo thứ tự cố định của cache final."""
        keras.utils.set_random_seed(seed)
        np.random.seed(seed)

        counts_original, order, inverse = sorted_class_mapping(y_original)
        y = inverse[y_original].astype(np.int64)
        counts = counts_original[order]
        supports = [X[y == i] for i in range(15)]  # lớp lớn nhất đứng cuối, không cần head sinh

        opt = keras.optimizers.legacy.Adam(
            learning_rate=CAU_HINH["gan_learning_rate"], beta_1=CAU_HINH["gan_beta1"]
        )
        C = tao_classifier(SHAPE, 16)
        C.compile(loss="mean_squared_error", optimizer=opt)
        D = tao_discriminator(SHAPE, 16)
        D.compile(loss="mean_squared_error", optimizer=opt)
        G = tao_generator(CAU_HINH["gan_latent_dim"], 16)
        heads = [tao_head_convex(s, SHAPE) for s in supports]

        # Composite G->head->C / D. Đóng băng C,D khi compile composite.
        C.trainable = False
        D.trainable = False
        vs_c, vs_d = [], []
        for i, head in enumerate(heads):
            z = keras.layers.Input(shape=(CAU_HINH["gan_latent_dim"],))
            lab = keras.layers.Input(shape=(16,))
            fake = head(G([z,lab]))
            mc = keras.Model([z,lab], C(fake), name=f"G_vs_C_{i}")
            mc.compile(loss="mean_squared_error", optimizer=opt)
            vs_c.append(mc)

            z2 = keras.layers.Input(shape=(CAU_HINH["gan_latent_dim"],))
            lab2 = keras.layers.Input(shape=(16,))
            lab3 = keras.layers.Input(shape=(16,))
            fake2 = head(G([z2,lab2]))
            md = keras.Model([z2,lab2,lab3], D([fake2,lab3]), name=f"G_vs_D_{i}")
            md.compile(loss="mean_squared_error", optimizer=opt)
            vs_d.append(md)
        C.trainable = True
        D.trainable = True

        # Slot cho toàn bộ shared Adam phải có trước khi restore checkpoint.
        all_variables = C.trainable_variables + D.trainable_variables + G.trainable_variables
        for head in heads:
            all_variables += head.trainable_variables
        _initialize_optimizer(opt, all_variables)

        return {
            "C":C,"D":D,"G":G,"heads":heads,"vs_c":vs_c,"vs_d":vs_d,
            "order":order,"inverse":inverse,"y":y,"counts":counts,"supports":supports,
            "optimizer":opt,
        }
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 19.3 Smoke: forward + bốn đường cập nhật + freeze + Adam clock
    def _smoke_snapshot(m):
        named = {"C": m["C"], "D": m["D"], "G": m["G"]}
        named.update({f"head_{i}": head for i, head in enumerate(m["heads"])})
        return {name: [w.numpy().copy() for w in model.weights] for name, model in named.items()}


    def _smoke_assert_update(m, action, changed_names, label):
        before = _smoke_snapshot(m)
        iteration_before = int(m["optimizer"].iterations.numpy())
        loss = _finite_loss(action(), label)
        after = _smoke_snapshot(m)
        if int(m["optimizer"].iterations.numpy()) != iteration_before+1:
            raise AssertionError(f"{label}: shared Adam phải tăng đúng một lượt.")
        for name in before:
            changed = any(not np.array_equal(a, b) for a, b in zip(before[name], after[name]))
            if name in changed_names and not changed:
                raise AssertionError(f"{label}: {name} không cập nhật.")
            if name not in changed_names and changed:
                raise AssertionError(f"{label}: {name} cập nhật ngoài ý muốn (freeze hỏng).")
        _finite_model_weights([m["C"], m["D"], m["G"]]+m["heads"])
        return {"action": label, "loss": loss, "optimizer_iteration": iteration_before+1,
                "changed_models": sorted(changed_names)}


    SMOKE_FILE = RUN/"hypergamo_smoke_pass.json"
    _smoke_record = doc_json(SMOKE_FILE) if SMOKE_FILE.exists() else None
    if _smoke_record is None or _smoke_record.get("smoke_version") != 2 or _smoke_record.get("tensorflow") != tf.__version__ or _smoke_record.get("gpu_present") != bool(tf.config.list_logical_devices("GPU")):
        keras.backend.clear_session()
        rng = np.random.default_rng(7)
        Xs = rng.normal(size=(34, *SHAPE)).astype(np.float32)
        ys = np.concatenate([np.repeat(np.arange(16), 2), np.array([15, 15])])
        m = build_hypergamo_models(Xs, ys, 7)
        lab = np.zeros((2, 16), np.float32); lab[:, 0] = 1
        z = rng.normal(size=(2, CAU_HINH["gan_latent_dim"])).astype(np.float32)
        snapshot_before_forward = _smoke_snapshot(m)
        feat = m["G"].predict_on_batch([z, lab])
        fake = m["heads"][0].predict_on_batch(feat)
        assert feat.shape == (2, 784) and fake.shape == (2, *SHAPE)
        assert np.isfinite(fake).all()
        snapshot_after_forward = _smoke_snapshot(m)
        assert all(np.array_equal(a, b) for name in snapshot_before_forward
            for a, b in zip(snapshot_before_forward[name], snapshot_after_forward[name]))
        real = m["supports"][0][:2]
        checks = [
            _smoke_assert_update(m, lambda: m["C"].train_on_batch(real, lab), {"C"}, "C_real"),
            _smoke_assert_update(m, lambda: m["D"].train_on_batch([real, lab], np.ones((2, 1), np.float32)), {"D"}, "D_real"),
            _smoke_assert_update(m, lambda: m["vs_c"][0].train_on_batch([z, lab], 1-lab), {"G", "head_0"}, "G_vs_C"),
            _smoke_assert_update(m, lambda: m["vs_d"][0].train_on_batch([z, lab, lab], np.ones((2, 1), np.float32)), {"G", "head_0"}, "G_vs_D"),
        ]
        # Verify optimizer slots, counters and every network can be restored before costly research training.
        import tempfile
        expected_snapshot = _smoke_snapshot(m)
        expected_optimizer = [w.copy() for w in m['optimizer'].get_weights()]
        smoke_objects = {name:m[name] for name in ['C','D','G']}
        smoke_objects['optimizer'] = m['optimizer']
        smoke_objects.update({f'head_{i:02d}':head for i,head in enumerate(m['heads'])})
        with tempfile.TemporaryDirectory(prefix='smoke_checkpoint_',dir=RUN) as smoke_dir:
            contract = {'kind':'synthetic_smoke','run_id':run_id}
            _save_recovery(Path(smoke_dir),smoke_objects,{'contract':contract,'updates':4})
            keras.backend.clear_session()
            restored_m = build_hypergamo_models(Xs,ys,7)
            restored_objects = {name:restored_m[name] for name in ['C','D','G']}
            restored_objects['optimizer'] = restored_m['optimizer']
            restored_objects.update({f'head_{i:02d}':head for i,head in enumerate(restored_m['heads'])})
            state = _restore_recovery(Path(smoke_dir),restored_objects,contract)
            actual_snapshot = _smoke_snapshot(restored_m)
            for name in expected_snapshot:
                for expected,actual in zip(expected_snapshot[name],actual_snapshot[name]):
                    np.testing.assert_array_equal(expected,actual)
            actual_optimizer=restored_m['optimizer'].get_weights()
            assert len(expected_optimizer)==len(actual_optimizer)
            for expected,actual in zip(expected_optimizer,actual_optimizer):
                np.testing.assert_array_equal(expected,actual)
            assert state['updates']==4
            del restored_m,restored_objects
        del expected_snapshot,expected_optimizer,smoke_objects,actual_snapshot,actual_optimizer

        luu_json(SMOKE_FILE, {"passed": True, "checkpoint_roundtrip":True, "smoke_version": 2,
            "run_id":run_id,"implementation":IMPLEMENTATION_SHA256,"tensorflow":tf.__version__,"tf_keras":keras.__version__,"gpu_present":bool(tf.config.list_logical_devices("GPU")),"generator_output": list(feat.shape), "fake_shape": list(fake.shape), "checks": checks})
        print("SMOKE PASSED: forward, C/D/G/head updates, inactive heads, freeze và shared Adam.")
        del m, Xs, ys, real, fake, feat, snapshot_before_forward, snapshot_after_forward
        keras.backend.clear_session()
    else:
        print("Đã có smoke test cập nhật đầy đủ:", _smoke_record)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 20. FINAL — 3D-HyperGAMO

Mỗi seed train **đúng 8000 classifier updates**. Không đánh giá validation để chọn step. Nếu phiên bị ngắt trước `done.json`, seed đó được chạy lại từ đầu; final run không nối resume để tránh RNG khác giữa các seed.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 20.1 HyperGAMO: source loop, 8000 C updates và recovery optimizer/state
    def save_hypergamo_weights(folder, m):
        paths = []
        for name in ["C", "D", "G"]:
            path = folder/f"{name}.weights.h5"
            m[name].save_weights(path); paths.append(path)
        heads_dir = folder/"heads"; heads_dir.mkdir(exist_ok=True)
        for i, head in enumerate(m["heads"]):
            path = heads_dir/f"head_{i:02d}.weights.h5"
            head.save_weights(path); paths.append(path)
        return paths


    def _hypergamo_checkpoint_objects(m):
        objects = {"C": m["C"], "D": m["D"], "G": m["G"], "optimizer": m["optimizer"]}
        objects.update({f"head_{i:02d}": head for i, head in enumerate(m["heads"])})
        return objects


    def train_hypergamo(seed):
        method = "hypergamo"
        folder = RUN/"models"/method/f"seed_{seed}"
        folder.mkdir(parents=True, exist_ok=True)
        required = ["C.weights.h5", "D.weights.h5", "G.weights.h5", "class_order.json"]
        required += [f"heads/head_{i:02d}.weights.h5" for i in range(15)]
        completed = load_done_checked(folder, required)
        if completed is not None:
            return completed
        if not _allow_training_mode("baseline", "HyperGAMO"):
            return None
        if not SMOKE_FILE.exists() or doc_json(SMOKE_FILE).get("smoke_version") != 2:
            raise RuntimeError("Cần chạy smoke cập nhật/freeze trước HyperGAMO.")
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        np.random.seed(seed)
        rng = np.random.default_rng(seed)
        m = build_hypergamo_models(X_final_train, y_final_train, seed)
        Y = keras.utils.to_categorical(m["y"], 16).astype(np.float32)
        n, bs = len(X_final_train), int(CAU_HINH["gan_batch_size"])
        target = int(CAU_HINH["gan_classifier_updates"])
        if target <= 0 or target % 2:
            raise ValueError("HyperGAMO budget phải là số chẵn dương.")
        n_batches, gen_points = int(math.ceil(n/bs)), int(math.ceil(bs/16))
        deficits = m["counts"][-1]-m["counts"][:-1]
        if np.any(deficits <= 0) or deficits.sum() <= 0:
            raise ValueError("Notebook này dùng 15 minority heads; cần một lớp majority duy nhất.")
        cdf = np.cumsum(deficits/deficits.sum()); cdf[-1] = 1.0
        n_heads, clock_per_loop = len(m["heads"]), 4+2*len(m["heads"])
        log_every = int(CAU_HINH["gan_log_every"])
        save_every = int(CAU_HINH.get("gan_checkpoint_every_updates", log_every))
        if log_every <= 0 or log_every % 2 or save_every <= 0 or save_every % 2:
            raise ValueError("GAN log/checkpoint cadence phải là số chẵn dương.")
        contract = {"method": method, "seed": int(seed), "target_updates": target,
            "lr": float(CAU_HINH["gan_learning_rate"]), "beta1": float(CAU_HINH["gan_beta1"]),
            "batch_size": bs, "samples": int(n), "heads": n_heads,
            "gen_points": gen_points, "clock_per_loop": clock_per_loop}
        objects = _hypergamo_checkpoint_objects(m)
        restored = _restore_recovery(folder, objects, contract)
        if restored:
            rng.bit_generator.state = restored["rng_state"]
            real_order = np.asarray(restored["real_order"], dtype=np.int64)
            updates, loop = int(restored["updates"]), int(restored["loop"])
            rows, recent, recent_n = restored["rows"], np.asarray(restored["recent"], dtype=float), int(restored["recent_n"])
            generated_counts = np.asarray(restored["generated_counts"], dtype=np.int64)
            elapsed_base = float(restored["elapsed_minutes"])
        else:
            real_order = rng.permutation(n)
            updates, loop, rows, recent_n, elapsed_base = 0, 0, [], 0, 0.0
            recent, generated_counts = np.zeros(4), np.zeros(16, dtype=np.int64)
        if updates != 2*loop or int(m["optimizer"].iterations.numpy()) != clock_per_loop*loop:
            raise RuntimeError("GAN counters không khớp recovery.")
        t0 = time.time()
        while updates < target:
            j = loop % n_batches
            idx = real_order[j*bs:min((j+1)*bs, n)]
            size = len(idx)
            valid = (1.0-rng.uniform(0, .1, size=(size, 1))).astype(np.float32)
            c_real = _finite_loss(m["C"].train_on_batch(X_final_train[idx], Y[idx]), "C_real")
            d_real = _finite_loss(m["D"].train_on_batch([X_final_train[idx], Y[idx]], valid), "D_real")
            fake_ids = np.searchsorted(cdf, rng.random(size), side="right")
            generated_counts += np.bincount(fake_ids, minlength=16)
            fake_labels = keras.utils.to_categorical(fake_ids, 16).astype(np.float32)
            z = rng.normal(size=(size, CAU_HINH["gan_latent_dim"])).astype(np.float32)
            feat = m["G"].predict_on_batch([z, fake_labels])
            fake = np.zeros((size, *SHAPE), dtype=np.float32)
            for i, head in enumerate(m["heads"]):
                ii = np.flatnonzero(fake_ids == i)
                if len(ii):
                    fake[ii] = head.predict_on_batch(feat[ii])
            if not np.isfinite(fake).all():
                raise FloatingPointError("Generated patch không hữu hạn.")
            invalid = rng.uniform(0, .1, size=(size, 1)).astype(np.float32)
            c_fake = _finite_loss(m["C"].train_on_batch(fake, fake_labels), "C_fake")
            d_fake = _finite_loss(m["D"].train_on_batch([fake, fake_labels], invalid), "D_fake")
            gc, gd = [], []
            for i in range(n_heads):
                cond = np.zeros((gen_points, 16), np.float32); cond[:, i] = 1
                noise = rng.normal(size=(gen_points, CAU_HINH["gan_latent_dim"])).astype(np.float32)
                gc.append(_finite_loss(m["vs_c"][i].train_on_batch([noise, cond], 1-cond), f"G_vs_C_{i}"))
                gd.append(_finite_loss(m["vs_d"][i].train_on_batch([noise, cond, cond], np.ones((gen_points, 1), np.float32)), f"G_vs_D_{i}"))
            updates, loop = updates+2, loop+1
            if int(m["optimizer"].iterations.numpy()) != clock_per_loop*loop:
                raise RuntimeError("Shared Adam phải nhận 4+2*n_heads calls mỗi GAN loop.")
            recent += [(c_real+c_fake)/2, (d_real+d_fake)/2, np.mean(gc), np.mean(gd)]
            recent_n += 1
            elapsed = elapsed_base+(time.time()-t0)/60
            if updates % log_every == 0 or updates == target:
                rows.append({"classifier_updates": updates, "optimizer_clock": int(m["optimizer"].iterations.numpy()),
                    "C_loss": float(recent[0]/recent_n), "D_loss": float(recent[1]/recent_n),
                    "G_vs_C_loss": float(recent[2]/recent_n), "G_vs_D_loss": float(recent[3]/recent_n),
                    "elapsed_minutes": elapsed})
                print("HyperGAMO seed", seed, rows[-1])
                recent[:] = 0; recent_n = 0
                pd.DataFrame(rows).to_csv(folder/"history.csv", index=False)
            if updates % save_every == 0 or updates == target:
                _save_recovery(folder, objects, {"contract": contract, "updates": updates,
                    "loop": loop, "real_order": real_order.tolist(), "rng_state": rng.bit_generator.state,
                    "rows": rows, "recent": recent.tolist(), "recent_n": recent_n,
                    "generated_counts": generated_counts.tolist(), "elapsed_minutes": elapsed})
        _finite_model_weights([m["C"], m["D"], m["G"]]+m["heads"])
        weight_paths = save_hypergamo_weights(folder, m)
        original_ids = (m["order"]+1).tolist()
        luu_json(folder/"class_order.json", {"original_class_ids": original_ids})
        luu_json(folder/"generated_counts.json", {"sorted_class_ids": original_ids,
            "generated_presentations": generated_counts.tolist(),
            "note": "Số lượt fake đưa vào C; không phải số mẫu unique."})
        meta = {"complete": True, "method": method, "seed": int(seed),
            "learning_rate": float(CAU_HINH["gan_learning_rate"]), "beta1": float(CAU_HINH["gan_beta1"]),
            "classifier_updates": target, "optimizer_clock": int(m["optimizer"].iterations.numpy()),
            "gan_loops": loop, "selection_rule": "fixed final C update budget; no checkpoint selection by validation/test",
            "elapsed_minutes": elapsed_base+(time.time()-t0)/60,
            "resume_policy": "optimizer/model/NumPy/batch state; not bitwise TF dropout"}
        stamp_done(folder, meta, weight_paths+[folder/"class_order.json"])
        return meta

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 20.2 Train HyperGAMO seed 42

In [ ]:
if CHE_DO_CHAY != 'preflight':
    train_hypergamo(42)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 20.3 Train HyperGAMO seed 43

In [ ]:
if CHE_DO_CHAY != 'preflight':
    train_hypergamo(43)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 20.4 Train HyperGAMO seed 44

In [ ]:
if CHE_DO_CHAY != 'preflight':
    train_hypergamo(44)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # 20.5 Trực quan loss của ba HyperGAMO seed
    fig, ax = plt.subplots(1,2,figsize=(14,4.5))
    for seed in CAU_HINH["seeds"]:
        p = RUN / "models" / "hypergamo" / f"seed_{seed}" / "history.csv"
        if p.exists():
            h = pd.read_csv(p)
            ax[0].plot(h.classifier_updates, h.C_loss, label=f"seed {seed}")
            ax[1].plot(h.classifier_updates, h.G_vs_C_loss, label=f"seed {seed}")
    ax[0].set_title("Classifier C loss")
    ax[1].set_title("Generator chống C")
    for a in ax:
        a.set_xlabel("Classifier updates"); a.grid(alpha=.2); a.legend()
    fig.tight_layout()
    luu_hinh(fig, "18_hypergamo_training_losses")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 21. Chẩn đoán 3D-HyperGAMO trước khi thiết kế cải tiến

Không được gọi `near-copy` chỉ vì một RMSE nhỏ. Notebook so sánh:

- mức tập trung trọng số `max_weight`;
- entropy và `effective_supports`;
- variance của fake so với real;
- khoảng cách fake → real;
- baseline real → real;
- độ đa dạng fake ↔ fake;
- trực quan patch, profile phổ/PCA và embedding 2D/3D.

Với lớp có nhiều support, full-patch pairwise distance rất tốn; notebook dùng full-patch RMSE cho lớp có `≤60` support và center-spectrum distance cho toàn bộ lớp.

In [ ]:
if CHE_DO_CHAY in {'diagnostics', 'test'}:
    # ============================================================
    # 21.1 HÀM LOAD HYPERGAMO ĐÃ TRAIN VÀ CHẨN ĐOÁN
    # ============================================================

    def load_hypergamo_final(seed):
        folder = RUN / "models" / "hypergamo" / f"seed_{seed}"
        if load_done_checked(folder, ["C.weights.h5", "D.weights.h5", "G.weights.h5"] + [f"heads/head_{i:02d}.weights.h5" for i in range(15)]) is None:
            raise RuntimeError(f"HyperGAMO seed {seed} chưa hoàn tất")
        keras.backend.clear_session()
        m = build_hypergamo_models(X_final_train, y_final_train, seed)
        m["C"].load_weights(folder / "C.weights.h5")
        m["D"].load_weights(folder / "D.weights.h5")
        m["G"].load_weights(folder / "G.weights.h5")
        for i, head in enumerate(m["heads"]):
            head.load_weights(folder / "heads" / f"head_{i:02d}.weights.h5")
        return m



    from scipy.spatial.distance import cdist

    def nearest_rmse_matrix(A, B, batch=64):
        A = np.asarray(A).reshape(len(A), -1).astype(np.float64)
        B = np.asarray(B).reshape(len(B), -1).astype(np.float64)
        return np.concatenate([np.sqrt(cdist(A[i:i+batch], B, 'sqeuclidean').min(1)/A.shape[1])
                               for i in range(0, len(A), batch)])

    def real_to_other_real_rmse(X):
        F = np.asarray(X).reshape(len(X), -1).astype(np.float64)
        d2 = cdist(F, F, 'sqeuclidean'); np.fill_diagonal(d2, np.inf)
        return np.sqrt(d2.min(1)/F.shape[1])

    def matched_diversity(real, fake, rng, repeats=32):
        n = min(len(real), len(fake), 64)
        rr, ff = [], []
        for _ in range(repeats):
            rr.append(float(np.median(real_to_other_real_rmse(real[rng.choice(len(real), n, replace=False)]))))
            ff.append(float(np.median(real_to_other_real_rmse(fake[rng.choice(len(fake), n, replace=False)]))))
        ratios = np.asarray(ff)/np.maximum(rr, 1e-12)
        return {'diversity_matched_n': n, 'diversity_repeats': repeats,
                'real_nn_matched_median': float(np.median(rr)),
                'fake_nn_matched_median': float(np.median(ff)),
                'diversity_nn_ratio_median': float(np.median(ratios)),
                'diversity_nn_ratio_p05': float(np.quantile(ratios, .05)),
                'diversity_nn_ratio_p95': float(np.quantile(ratios, .95))}

    def run_gan_diagnostics(seed):
        folder = RUN / "diagnostics" / f"hypergamo_seed_{seed}"
        folder.mkdir(parents=True, exist_ok=True)
        done = folder / "done.json"
        if load_done_checked(folder, ["summary.csv", "samples.npz"]) is not None:
            print("Đã có diagnostics seed", seed)
            return pd.read_csv(folder / "summary.csv")

        m = load_hypergamo_final(seed)
        rng = np.random.default_rng(1701 + seed)
        n_gen = CAU_HINH["diagnostic_generated_per_class"]
        n_save = CAU_HINH["diagnostic_saved_per_class"]
        rows = []
        saved = {}

        for i, head in enumerate(m["heads"]):
            original_id = int(m["order"][i] + 1)
            support = m["supports"][i]
            fake_batches, weight_batches = [], []
            # Lấy layer softmax weights của head.
            weight_model = keras.Model(head.input, head.get_layer("support_weights").output)
            for start in range(0, n_gen, 128):
                bs = min(128, n_gen-start)
                cond = np.zeros((bs,16),np.float32); cond[:,i] = 1
                z = rng.normal(size=(bs,CAU_HINH["gan_latent_dim"]))
                feat = m["G"].predict_on_batch([z,cond])
                fake_batches.append(head.predict_on_batch(feat))
                weight_batches.append(weight_model.predict_on_batch(feat))
            fake = np.concatenate(fake_batches)
            weights = np.concatenate(weight_batches)

            entropy = -(weights*np.log(np.maximum(weights,1e-12))).sum(1)
            norm_entropy = entropy / np.log(len(support)) if len(support) > 1 else np.zeros_like(entropy)
            effective = np.exp(entropy)
            max_weight = weights.max(1)
            variance_ratio = float(fake.var(axis=0).mean() / (support.var(axis=0).mean()+1e-12))
            matched = matched_diversity(support[:,CAU_HINH["patch_size"]//2,CAU_HINH["patch_size"]//2,:,0],
                                        fake[:,CAU_HINH["patch_size"]//2,CAU_HINH["patch_size"]//2,:,0], rng)

            # Center spectrum/PCA vector: 40 chiều ở pixel tâm.
            c = CAU_HINH["patch_size"]//2
            fake_center = fake[:,c,c,:,0]
            real_center = support[:,c,c,:,0]
            fake_center_rmse = nearest_rmse_matrix(fake_center, real_center)
            real_center_baseline = real_to_other_real_rmse(real_center) if len(real_center)>1 else np.array([np.nan])

            full_fake_rmse_mean = np.nan
            full_real_rmse_mean = np.nan
            if len(support) <= 60 and len(support) > 1:
                full_fake = nearest_rmse_matrix(fake[:min(256,len(fake))], support)
                full_real = real_to_other_real_rmse(support)
                full_fake_rmse_mean = float(full_fake.mean())
                full_real_rmse_mean = float(full_real.mean())

            # Fake-fake diversity trên center vector: khoảng cách tới fake khác gần nhất.
            fake_fake = real_to_other_real_rmse(fake_center[:min(512,len(fake_center))])

            # Train-only controls: uniform random convex mixture, random copy and class mean.
            random_w = rng.dirichlet(np.ones(len(support)), size=n_gen).astype(np.float64)
            random_center = random_w @ real_center.astype(np.float64)
            random_div = matched_diversity(real_center, random_center, rng)
            copy_center = real_center[rng.integers(len(real_center), size=n_gen)]
            rows.append({
                "class_id": original_id,
                "class_name": TEN_LOP[original_id-1],
                "real_support": len(support),
                **matched,
                "mean_max_weight": float(max_weight.mean()),
                "weight_variance_across_noise": float(weights.var(axis=0).sum()),
                "dominant_supports_used": int(len(np.unique(weights.argmax(1)))),
                "p95_max_weight": float(np.quantile(max_weight,.95)),
                "normalized_entropy": float(norm_entropy.mean()),
                "effective_supports": float(effective.mean()),
                "variance_ratio": variance_ratio,
                "fake_to_real_center_rmse": float(fake_center_rmse.mean()),
                "real_to_real_center_rmse": float(np.nanmean(real_center_baseline)),
                "fake_to_fake_center_rmse": float(fake_fake.mean()),
                "random_convex_variance_ratio_center": float(random_center.var(0).mean()/(real_center.var(0).mean()+1e-12)),
                "random_convex_to_real_center_rmse": float(nearest_rmse_matrix(random_center,real_center).mean()),
                "random_convex_diversity_nn_ratio": random_div["diversity_nn_ratio_median"],
                "copy_to_real_center_rmse": float(nearest_rmse_matrix(copy_center,real_center).mean()),
                "class_mean_to_real_center_rmse": float(nearest_rmse_matrix(real_center.mean(0,keepdims=True),real_center).mean()),
                "fake_to_real_full_patch_rmse": full_fake_rmse_mean,
                "real_to_real_full_patch_rmse": full_real_rmse_mean,
            })

            saved[f"fake_class_{original_id}"] = fake[:n_save]
            saved[f"weights_class_{original_id}"] = weights[:n_save]
            saved[f"support_class_{original_id}"] = support

        summary = pd.DataFrame(rows).sort_values("real_support")
        summary.to_csv(folder / "summary.csv", index=False)
        np.savez_compressed(folder / "samples.npz", **saved)
        stamp_done(folder, {"seed":seed,"generated_per_class":n_gen,"saved_per_class":n_save}, ["summary.csv","samples.npz"])
        return summary

    if CHE_DO_CHAY not in {'diagnostics', 'test'}:
        print('Chưa chạy diagnostics trong chế độ này.')

    all_diag = []
    for seed in CAU_HINH["seeds"]:
        d = run_gan_diagnostics(seed)
        d.insert(0,"seed",seed)
        all_diag.append(d)
    diag_all = pd.concat(all_diag, ignore_index=True)
    diag_all.to_csv(RUN / "tables" / "hypergamo_diagnostics_all_seeds.csv", index=False)
    display(diag_all.head(20))

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY in {'diagnostics', 'test'}:
    # ============================================================
    # 21.2 TỔNG HỢP DIAGNOSTICS THEO LỚP
    # ============================================================

    diag_mean = diag_all.groupby(["class_id","class_name"], as_index=False).agg({
        "real_support":"mean",
        "mean_max_weight":"mean",
        "normalized_entropy":"mean",
        "effective_supports":"mean",
        "variance_ratio":"mean",
        "fake_to_real_center_rmse":"mean",
        "real_to_real_center_rmse":"mean",
        "fake_to_fake_center_rmse":"mean",
        "fake_to_real_full_patch_rmse":"mean",
        "real_to_real_full_patch_rmse":"mean",
    }).sort_values("real_support")
    diag_mean.to_csv(RUN / "tables" / "hypergamo_diagnostics_mean.csv", index=False)
    display(diag_mean)

    fig, ax = plt.subplots(1,3,figsize=(18,4.5))
    ax[0].bar(diag_mean.class_id, diag_mean.mean_max_weight)
    ax[0].set_title("Max generator weight trung bình")
    ax[1].bar(diag_mean.class_id, diag_mean.effective_supports)
    ax[1].set_title("Effective supports")
    ax[2].bar(diag_mean.class_id, diag_mean.variance_ratio)
    ax[2].set_title("Variance fake / real")
    for a in ax:
        a.set_xticks(diag_mean.class_id); a.set_xlabel("ID lớp"); a.grid(axis="y",alpha=.2)
    fig.tight_layout()
    luu_hinh(fig, "19_hypergamo_diagnostics_summary")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY in {'diagnostics', 'test'}:
    # ============================================================
    # 21.3 TRỰC QUAN CHI TIẾT CÁC LỚP CỰC ÍT MẪU
    # ============================================================

    CRITICAL = [9, 7, 1, 16]  # Oats, Grass-pasture-mowed, Alfalfa, Stone-Steel-Towers
    seed_show = 42
    sample_file = RUN / "diagnostics" / f"hypergamo_seed_{seed_show}" / "samples.npz"
    with np.load(sample_file, allow_pickle=False) as d:
        for cid in CRITICAL:
            key = f"fake_class_{cid}"
            if key not in d.files:
                continue
            fake = d[key]
            support = d[f"support_class_{cid}"]
            weights = d[f"weights_class_{cid}"]
            c = CAU_HINH["patch_size"]//2

            # Patch thật gần nhất theo center PCA vector để minh họa.
            center_fake = fake[0,c,c,:,0]
            center_real = support[:,c,c,:,0]
            j = np.argmin(((center_real-center_fake)**2).mean(1))
            real = support[j]

            fig, ax = plt.subplots(2,3,figsize=(13,7))
            for k, pc in enumerate([0,4,9]):
                ax[0,k].imshow(real[...,pc,0], cmap="viridis")
                ax[0,k].set_title(f"Real gần — PC{pc+1}")
                ax[1,k].imshow(fake[0,...,pc,0], cmap="viridis")
                ax[1,k].set_title(f"Fake — PC{pc+1}")
                ax[0,k].axis("off"); ax[1,k].axis("off")
            fig.suptitle(f"Lớp {cid}: {TEN_LOP[cid-1]} — real vs fake")
            fig.tight_layout()
            luu_hinh(fig, f"20_real_fake_class_{cid}")

            top = np.argsort(weights[0])[::-1][:10]
            fig, ax = plt.subplots(figsize=(9,4))
            ax.bar(np.arange(len(top)), weights[0][top])
            ax.set_xticks(np.arange(len(top)))
            ax.set_xticklabels([str(int(t)) for t in top])
            ax.set_xlabel("Chỉ số support thật")
            ax.set_ylabel("Trọng số softmax")
            ax.set_title(f"Top-10 generator weights — lớp {cid}")
            ax.grid(axis="y",alpha=.2)
            fig.tight_layout()
            luu_hinh(fig, f"21_weights_class_{cid}")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY in {'diagnostics', 'test'}:
    # ============================================================
    # 21.4 PCA EMBEDDING 2D/3D: REAL vs SYNTHETIC
    # ============================================================

    cid = 9  # Oats mặc định vì cực ít; có thể đổi nhưng không dùng hình này làm metric chính.
    with np.load(RUN / "diagnostics" / "hypergamo_seed_42" / "samples.npz", allow_pickle=False) as d:
        fake = d[f"fake_class_{cid}"]
        real = d[f"support_class_{cid}"]

    c = CAU_HINH["patch_size"]//2
    A = real[:,c,c,:,0]
    B = fake[:,c,c,:,0]
    emb = PCA(n_components=3).fit_transform(np.vstack([A,B]))
    real_e, fake_e = emb[:len(A)], emb[len(A):]

    fig, ax = plt.subplots(figsize=(7,6))
    ax.scatter(real_e[:,0], real_e[:,1], label="Real")
    ax.scatter(fake_e[:,0], fake_e[:,1], marker="x", label="Synthetic")
    ax.set_xlabel("PCA embedding 1"); ax.set_ylabel("PCA embedding 2")
    ax.set_title(f"Real vs synthetic — lớp {cid}: {TEN_LOP[cid-1]}")
    ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout()
    luu_hinh(fig, f"22_embedding_2d_class_{cid}")

    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(real_e[:,0],real_e[:,1],real_e[:,2],label="Real")
    ax.scatter(fake_e[:,0],fake_e[:,1],fake_e[:,2],marker="x",label="Synthetic")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
    ax.set_title(f"Embedding 3D — lớp {cid}")
    ax.legend()
    fig.tight_layout()
    luu_hinh(fig, f"23_embedding_3d_class_{cid}")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### Cách diễn giải diagnostics

Khoảng cách dùng float64 ổn định. Fake–fake và real–real được lấy cùng số lượng rồi
lặp 32 lần; báo median và khoảng 5–95% qua các lần lấy mẫu. Thêm đối chứng random-copy,
class mean và tổ hợp lồi Dirichlet cùng supports. Đây là diagnostics trên **train**.

Variance ratio thấp có thể do tính chất tổ hợp lồi; entropy cao từng mẫu chưa bảo đảm
đa dạng theo noise. Near-copy/variance nhỏ riêng lẻ chưa chứng minh generator hỏng;
cần đồng thời xem phân bố weights, đa dạng, đối chứng và tác động phân loại qua ablation.


## 22. Phần cải tiến — cố ý chưa tự bịa trước khi có bằng chứng

Đây là phần duy nhất notebook **không tự chốt phương pháp trước**. Sau khi chạy diagnostics ở mục 21, ta sẽ chọn đúng một hạn chế đã được chứng minh và cài một cải tiến tương ứng.

Khi chốt cải tiến, cần ghi vào đây:

1. **Hạn chế quan sát được** — chỉ số/lớp nào cho thấy vấn đề.
2. **Cơ chế dự kiến** — thay đổi generator/head/loss/sampling ở đâu.
3. **Giả thuyết kiểm chứng được** — chỉ số nào phải cải thiện.
4. **Ablation** — chỉ thay đúng thành phần đó so với HyperGAMO baseline.
5. **Ngân sách train** — giữ 8000 classifier updates và cùng master train/test nếu có thể.

> Không cài “diversity loss” hay một kỹ thuật bất kỳ chỉ vì nghe hợp lý. Phương pháp cải tiến sẽ được thêm vào notebook này sau khi mục 21 cho bằng chứng đủ mạnh.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # Khóa quyết định nghiên cứu sau diagnostics; giữ None trước bước này.
    # Hai schema hợp lệ:
    # {'decision': 'baseline_only', 'reason': 'Lý do khoa học cụ thể...'}
    # {'decision': 'include_improvement', 'reason': '...', 'methods': [
    #     {'method': 'hypergamo_improved', 'display_name': 'Tên cải tiến',
    #      'weights_file': 'C.weights.h5', 'class_order_file': 'class_order.json',
    #      'spec_file': 'improvement_spec.json', 'classifier_architecture': 'shared_C'}]}
    # Với include_improvement: từng model/seed phải có done.json đúng contract + hash weights;
    # spec_file được lưu ở RUN và khóa trước test. Cải tiến phải giữ architecture C dùng chung.
    print('Quyết định cải tiến:', PHUONG_PHAP_CAI_TIEN)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 23. Kiểm tra trạng thái trước khi mở test

Test chỉ được mở khi:

- ba CNN final đủ 3 seed;
- `C-real-only matched` đủ 3 seed;
- HyperGAMO đủ 3 seed;
- diagnostics hoàn tất;
- phương pháp cải tiến đã được định nghĩa và hoàn tất;
- cấu hình cuối đã khóa.

Trong giai đoạn baseline/diagnostics, `CHO_PHEP_MO_TEST=False` là trạng thái đúng.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    BASE_REGISTRY = [
        {'method':'cnn_thuong','display_name':'CNN thường','weights_file':'classifier.weights.h5'},
        {'method':'ros','display_name':'CNN + ROS','weights_file':'classifier.weights.h5'},
        {'method':'class_weight','display_name':'CNN + Class Weight','weights_file':'classifier.weights.h5'},
        {'method':'c_real_only_matched','display_name':'C-real-only clock matched','weights_file':'classifier.weights.h5', 'class_order_file':'class_order.json'},
        {'method':'hypergamo','display_name':'3D-HyperGAMO','weights_file':'C.weights.h5','class_order_file':'class_order.json'},
    ]
    def frozen_registry():
        decision = PHUONG_PHAP_CAI_TIEN
        if not isinstance(decision, dict) or not str(decision.get('reason', '')).strip():
            raise RuntimeError('Chưa khóa quyết định cải tiến có lý do. {} không hợp lệ.')
        registry = [dict(v) for v in BASE_REGISTRY]
        if decision.get('decision') == 'baseline_only':
            return registry
        if decision.get('decision') != 'include_improvement' or not decision.get('methods'):
            raise RuntimeError('Schema quyết định cải tiến không hợp lệ.')
        for item in decision['methods']:
            required = {'method','display_name','weights_file','spec_file','classifier_architecture'}
            if not required.issubset(item) or item['classifier_architecture'] != 'shared_C':
                raise RuntimeError('Cải tiến thiếu spec/weights hoặc thay C: cần loader và ablation riêng.')
            rel = Path(item['spec_file'])
            if rel.is_absolute() or '..' in rel.parts or not (RUN/rel).is_file():
                raise RuntimeError('Thiếu improvement spec lưu trong RUN.')
            registry.append(dict(item))
        if len({r['method'] for r in registry}) != len(registry):
            raise RuntimeError('Registry trùng method.')
        return registry

    status_rows = []
    for item in BASE_REGISTRY:
        for seed in CAU_HINH['seeds']:
            folder = RUN/'models'/item['method']/f'seed_{seed}'
            required = [item['weights_file']] + ([item['class_order_file']] if 'class_order_file' in item else [])
            meta = load_done_checked(folder, required)
            status_rows.append({'method':item['method'],'seed':seed,'complete':meta is not None})
    display(pd.DataFrame(status_rows))
    print('Test gate:', CHO_PHEP_MO_TEST, '| decision:', PHUONG_PHAP_CAI_TIEN)

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 24. Hàm đánh giá test — chỉ chạy khi cổng test được mở

Cell định nghĩa hàm phía dưới **không đọc test ngay**. Chỉ các cell evaluate riêng từng phương pháp mới đọc `test_4721_LOCKED.npz`, và chỉ khi `CHO_PHEP_MO_TEST=True`.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    def require_test_open():
        if CHE_DO_CHAY != 'test' or not CHO_PHEP_MO_TEST:
            raise RuntimeError('TEST ĐANG KHÓA: cần chế độ test + quyết định đã khóa.')
        registry = frozen_registry()
        snapshot = {'run_id':run_id,'implementation':IMPLEMENTATION_SHA256,
                    'decision':PHUONG_PHAP_CAI_TIEN,'registry':registry,'models':{},'diagnostics':{}}
        for item in registry:
            for seed in CAU_HINH['seeds']:
                folder = RUN/'models'/item['method']/f'seed_{seed}'
                required = [item['weights_file']] + ([item['class_order_file']] if 'class_order_file' in item else [])
                if load_done_checked(folder, required) is None:
                    raise RuntimeError(f'Model chưa hoàn tất: {folder}')
                snapshot['models'][f'{item["method"]}/{seed}'] = sha256_file(folder/'done.json')
            if 'spec_file' in item:
                snapshot.setdefault('improvement_specs',{})[item['spec_file']] = sha256_file(RUN/item['spec_file'])
        for seed in CAU_HINH['seeds']:
            folder = RUN/'diagnostics'/f'hypergamo_seed_{seed}'
            if load_done_checked(folder, ['summary.csv','samples.npz']) is None:
                raise RuntimeError('Diagnostics chưa hoàn tất.')
            snapshot['diagnostics'][str(seed)] = sha256_file(folder/'done.json')
        immutable_json(RUN/'final_test_freeze.json',snapshot)
        return registry

    def load_test_data():
        require_test_open()
        contract = cache_contract(kind='locked_test',coords=array_sha(coords_locked_test),
                                  preprocessing_sha256=sha256_file(FINAL_PREP))
        if not LOCKED_TEST_CACHE.exists():
            save_npz_checked(LOCKED_TEST_CACHE,contract,
                X_test=trich_patch(processed_final,coords_locked_test,CAU_HINH['patch_size']),
                y_test=gt[coords_locked_test[:,0],coords_locked_test[:,1]].astype(np.int64)-1,
                coords_test=coords_locked_test)
        verify_npz(LOCKED_TEST_CACHE,contract)
        with np.load(LOCKED_TEST_CACHE,allow_pickle=False) as d:
            np.testing.assert_array_equal(d['coords_test'],coords_locked_test)
            return d['X_test'],d['y_test'],d['coords_test']

    def load_final_classifier(method,seed):
        registry = frozen_registry()
        item = next((v for v in registry if v['method']==method),None)
        if item is None:
            raise ValueError(method)
        folder=RUN/'models'/method/f'seed_{seed}'
        required=[item['weights_file']] + ([item['class_order_file']] if 'class_order_file' in item else [])
        if load_done_checked(folder,required) is None:
            raise RuntimeError('Model chưa hoàn tất.')
        keras.backend.clear_session(); keras.utils.set_random_seed(seed)
        model=tao_classifier(SHAPE,16); model.load_weights(folder/item['weights_file'])
        order=np.array(doc_json(folder/item['class_order_file'])['original_class_ids'])-1 if 'class_order_file' in item else np.arange(16)
        np.testing.assert_array_equal(np.sort(order),np.arange(16))
        return model,order

    def evaluate_method_seed(method,display_name,seed):
        require_test_open()
        outdir=RUN/'test_results'/method/f'seed_{seed}'; outdir.mkdir(parents=True,exist_ok=True)
        done=load_done_checked(outdir,['metrics.json','per_class.csv','predictions.npz'])
        if done is not None:
            return done
        X_test,y_standard,coords=load_test_data()
        model,class_order=load_final_classifier(method,seed)
        p=model.predict(X_test,batch_size=CAU_HINH['eval_batch_size'],verbose=0)
        assert np.isfinite(p).all()
        prob=np.zeros_like(p); prob[:,class_order]=p
        summary,per_class,pred=bao_cao_metrics(y_standard,prob,16)
        luu_json(outdir/'metrics.json',summary); per_class.to_csv(outdir/'per_class.csv',index=False)
        np.savez_compressed(outdir/'predictions.npz',coords=coords,y_true=y_standard,y_pred=pred,
                            probabilities=prob,confusion_matrix=confusion_matrix(y_standard,pred,labels=np.arange(16)))
        return stamp_done(outdir,{'method':method,'display_name':display_name,'seed':seed,**summary},
                          ['metrics.json','per_class.csv','predictions.npz'])

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 24.2 TEST — CNN thường

In [ ]:
if CHE_DO_CHAY != 'preflight':
    if CHO_PHEP_MO_TEST:
        for seed in CAU_HINH["seeds"]:
            evaluate_method_seed("cnn_thuong","CNN thường",seed)
    else:
        print("Test đang khóa — bỏ qua CNN thường.")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 24.3 TEST — CNN + ROS

In [ ]:
if CHE_DO_CHAY != 'preflight':
    if CHO_PHEP_MO_TEST:
        for seed in CAU_HINH["seeds"]:
            evaluate_method_seed("ros","CNN + ROS",seed)
    else:
        print("Test đang khóa — bỏ qua ROS.")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 24.4 TEST — CNN + Class Weight

In [ ]:
if CHE_DO_CHAY != 'preflight':
    if CHO_PHEP_MO_TEST:
        for seed in CAU_HINH["seeds"]:
            evaluate_method_seed("class_weight","CNN + Class Weight",seed)
    else:
        print("Test đang khóa — bỏ qua Class Weight.")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 24.5 TEST — C-real-only matched

In [ ]:
if CHE_DO_CHAY != 'preflight':
    if CHO_PHEP_MO_TEST:
        for seed in CAU_HINH["seeds"]:
            evaluate_method_seed("c_real_only_matched","C-real-only matched",seed)
    else:
        print("Test đang khóa — bỏ qua matched ablation.")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


### 24.6 TEST — 3D-HyperGAMO

In [ ]:
if CHE_DO_CHAY != 'preflight':
    if CHO_PHEP_MO_TEST:
        for seed in CAU_HINH["seeds"]:
            evaluate_method_seed("hypergamo","3D-HyperGAMO",seed)
    else:
        print("Test đang khóa — bỏ qua HyperGAMO.")
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 25. Tổng hợp kết quả test và trực quan

Section này chỉ sinh bảng/hình khi test đã được mở. Mỗi hình được lưu vào `figures/` và bảng vào `tables/`, nên có thể dùng lại trong báo cáo mà không train lại.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 25.1 BẢNG MEAN ± SD
    # ============================================================

    if CHO_PHEP_MO_TEST:
        names = {item["method"]:item["display_name"] for item in require_test_open()}
        for item in require_test_open():
            if item['method'] not in {r['method'] for r in BASE_REGISTRY}:
                for seed in CAU_HINH['seeds']:
                    evaluate_method_seed(item['method'],item['display_name'],seed)
        rows = []
        for method, name in names.items():
            for seed in CAU_HINH["seeds"]:
                p = RUN / "test_results" / method / f"seed_{seed}" / "metrics.json"
                if p.exists():
                    rows.append({"method":name,"seed":seed,**doc_json(p)})
        assert len(rows) == len(names)*len(CAU_HINH["seeds"]), "Thiếu kết quả: không tổng hợp partial"
        test_by_seed = pd.DataFrame(rows)
        test_by_seed.to_csv(RUN / "tables" / "test_by_seed.csv", index=False)
        display(test_by_seed)

        summary_rows = []
        for method, g in test_by_seed.groupby("method"):
            row = {"method":method}
            for m in ["OA","AA","Macro_F1","GM","Kappa"]:
                row[m] = f"{g[m].mean()*100:.3f} ± {g[m].std(ddof=1)*100:.3f}" if m != "Kappa" else f"{g[m].mean():.4f} ± {g[m].std(ddof=1):.4f}"
            summary_rows.append(row)
        test_summary = pd.DataFrame(summary_rows)
        test_summary.to_csv(RUN / "tables" / "test_summary_mean_sd.csv", index=False)
        display(test_summary)
    else:
        print("Chưa mở test nên chưa có bảng kết quả cuối.")

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 25.2 HEATMAP F1 THEO LỚP + CONFUSION MATRIX + ERROR MAP
    # ============================================================

    if CHO_PHEP_MO_TEST:
        names = {item["method"]:item["display_name"] for item in require_test_open()}
        # F1 trung bình theo lớp.
        matrix = []
        for method in names:
            per_seed = []
            for seed in CAU_HINH["seeds"]:
                p = RUN / "test_results" / method / f"seed_{seed}" / "per_class.csv"
                per_seed.append(pd.read_csv(p).f1.to_numpy())
            matrix.append(np.mean(per_seed,axis=0))
        matrix = np.asarray(matrix)

        fig, ax = plt.subplots(figsize=(14,5))
        im = ax.imshow(matrix, aspect="auto", vmin=0, vmax=1, cmap="viridis")
        ax.set_yticks(np.arange(len(names))); ax.set_yticklabels(list(names.values()))
        ax.set_xticks(np.arange(16)); ax.set_xticklabels(ID_LOP)
        ax.set_xlabel("ID lớp"); ax.set_title("F1 từng lớp — trung bình 3 seed")
        fig.colorbar(im, ax=ax, label="F1")
        fig.tight_layout(); luu_hinh(fig,"24_test_per_class_f1_heatmap")

        # Confusion matrix seed42 riêng từng method để hình không bị quá nhỏ.
        for method, name in names.items():
            pred_file = RUN / "test_results" / method / "seed_42" / "predictions.npz"
            with np.load(pred_file, allow_pickle=False) as d:
                cm = d["confusion_matrix"]
                coords = d["coords"]
                y_true = d["y_true"]
                y_pred = d["y_pred"]
            fig, ax = plt.subplots(figsize=(7,6))
            im = ax.imshow(cm, cmap="viridis")
            ax.set_xlabel("Predicted class index")
            ax.set_ylabel("True class index")
            ax.set_title(name + " — confusion matrix, seed42")
            fig.colorbar(im, ax=ax)
            fig.tight_layout(); luu_hinh(fig,f"25_confusion_{method}_seed42")

            # Error map: chỉ test coordinates, không tô train bằng GT để tránh gây hiểu nhầm.
            error = np.full(gt.shape, np.nan)
            error[coords[:,0],coords[:,1]] = (y_true != y_pred).astype(float)
            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(error, cmap="viridis", vmin=0, vmax=1)
            ax.set_title(name + " — bản đồ lỗi test (seed42)")
            ax.axis("off")
            fig.tight_layout(); luu_hinh(fig,f"26_error_map_{method}_seed42")
    else:
        print("Chưa mở test nên chưa vẽ hình kết quả cuối.")

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 26. Bản đồ phân loại toàn cảnh — chỉ là minh họa sau test

Nếu cần hình trực quan kiểu paper, có thể dùng model seed42 dự đoán tất cả **pixel có nhãn Standard GT**. Hình này không dùng để tính thêm metric và không thay thế locked test.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    if CHO_PHEP_MO_TEST:
        registry=require_test_open()
        coords_all=np.argwhere(gt>0).astype(np.int16)
        maps=[]
        for item in registry:
            model,order=load_final_classifier(item['method'],42)
            predicted=[]
            for start in range(0,len(coords_all),128):
                x=trich_patch(processed_final,coords_all[start:start+128],CAU_HINH['patch_size'])
                p=model.predict_on_batch(x)
                predicted.extend((order[p.argmax(1)]+1).tolist())
            mp=np.zeros_like(gt); mp[coords_all[:,0],coords_all[:,1]]=predicted; maps.append(mp)
        n=len(maps)+1; fig,axes=plt.subplots(math.ceil(n/3),3,figsize=(16,5*math.ceil(n/3)))
        axes=np.asarray(axes).ravel(); axes[0].imshow(gt,cmap='tab20',vmin=0,vmax=16);axes[0].set_title('GT — illustration only')
        for ax,mp,item in zip(axes[1:],maps,registry):
            ax.imshow(mp,cmap='tab20',vmin=0,vmax=16);ax.set_title(item['display_name']+' — seed42')
        for ax in axes: ax.axis('off')
        fig.tight_layout();luu_hinh(fig,'27_full_labeled_scene_maps')

else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


## 27. Lưu, đóng gói và dùng lại kết quả

Notebook cố ý lưu:

- split + hash + protocol spec;
- preprocessing development/final;
- tuning histories + cấu hình CNN đã chọn;
- weights + history từng method/seed;
- diagnostics GAN;
- bảng CSV;
- toàn bộ figure PNG;
- test prediction/metrics khi test được mở.

Nhờ vậy có thể tắt phiên Kaggle rồi dùng lại kết quả mà không phải train lại.

In [ ]:
if CHE_DO_CHAY != 'preflight':
    # ============================================================
    # 27.1 TẠO TRANG HTML TÓM TẮT VÀ ZIP TOÀN BỘ RUN
    # ============================================================

    import html

    html_parts = [
        "<!doctype html><meta charset='utf-8'>",
        "<body style='font-family:Arial;max-width:1250px;margin:30px auto'>",
        "<h1>KLTN — 3D-HyperGAMO Protocol B</h1>",
        f"<p>Run: <b>{html.escape(RUN.name)}</b></p>",
        "<p>Master split: 5528 train / 4721 locked test; 16 lớp; patch 11×11; PCA40.</p>",
    ]

    for csv_path in sorted((RUN/"tables").glob("*.csv")):
        try:
            df = pd.read_csv(csv_path)
            html_parts.append(f"<h2>{html.escape(csv_path.stem)}</h2>")
            html_parts.append(df.to_html(index=False))
        except Exception:
            pass

    for image_path in sorted((RUN/"figures").glob("*.png")):
        rel = image_path.relative_to(RUN).as_posix()
        html_parts.append(f"<h2>{html.escape(image_path.stem)}</h2>")
        html_parts.append(f"<img style='max-width:100%' src='{html.escape(rel)}'>")

    html_parts.append("</body>")
    (RUN / "BAO_CAO_HINH.html").write_text("\n".join(html_parts), encoding="utf-8")

    zip_path = THU_MUC_KET_QUA / f"{RUN.name}.zip"
    tmp = zip_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for p in sorted(RUN.rglob("*")):
            if p.is_file() and not p.name.endswith(".tmp"):
                z.write(p, str(Path(RUN.name) / p.relative_to(RUN)))
    tmp.replace(zip_path)

    print("Đã lưu HTML:", RUN / "BAO_CAO_HINH.html")
    print("Đã đóng gói ZIP:", zip_path)
else:
    print('Section này chưa chạy trong chế độ', CHE_DO_CHAY)


# Ghi chú cuối

### Những gì Protocol B đã sửa so với A

- Loại bất đồng nhãn giữa DIP mask và Standard GT → 5528/4721.
- Không dùng validation làm kết quả generalization.
- CNN chỉ tuning trên seed42, sau đó dùng LR/số bước cập nhật cố định cho 3 seed final.
- Final CNN sử dụng lại toàn bộ 5528 train.
- HyperGAMO dùng fixed 8000 updates thay vì chọn checkpoint tốt nhất trên validation.
- Test được khóa đến cuối và training functions không đọc file test.
- Thêm `C-real-only matched` để giảm confound loss/optimizer/training loop.
- Chẩn đoán GAN lớn hơn, có real→real baseline chứ không chỉ nhìn fake→real.
- Toàn bộ kết quả/figure/config được lưu để tái sử dụng.

### Hạn chế vẫn còn và phải báo cáo

- Patch 11×11 của train/validation/test vẫn chồng lấn raw pixels trong cùng scene.
- Ba seed dùng cùng master split nên độ lệch chuẩn chỉ phản ánh stochastic training, không phản ánh split variability.
- Indian Pines là một scene nhỏ; kết luận không được mở rộng thành khả năng tổng quát sang cảnh hoàn toàn mới.
- Phương pháp cải tiến chỉ được chốt sau khi diagnostics baseline cho bằng chứng đủ mạnh.

### Đối chiếu paper và mã tác giả

Đây là đánh giá có kiểm soát dựa trên 3D-HyperGAMO. B thay label rule,
fit PCA/MinMax trên tâm train và dùng checkpoint cuối thay best-test trong repo.
Không gọi là tái lập nguyên trạng Table II. CNN3D75.09 trong Roy dùng baseline khác,
không phải mức dự kiến của CNN cùng kiến trúc C ở đây. CNN có 4 LR development,
GAN baseline giữ LR paper: không tuyên bố cùng ngân sách tuning giữa mọi pipeline.

- [Roy et al., 3D-HyperGAMO](https://profile.iiita.ac.in/srdubey/publications/2022_3D-HyperGAMO.pdf)
- [Mã tác giả được ghim](https://github.com/mhaut/3D-HyperGAMO/tree/3f92889aec4a7e2c8f7a2db4c8a9bf167f296f5e)
- [TransHSI, Table3](https://www.mdpi.com/2072-4292/15/22/5331)
- [DeepHyperX/Audebert](https://arxiv.org/pdf/1904.10674)
- [Liang et al., spatial sampling](https://arxiv.org/abs/1605.05829)

Checkpoint khi restart khôi phục weights, optimizer và NumPy state; không hứa
bitwise-identical TensorFlow Dropout. Lượt final liên tục ưu tiên nếu GPU cho phép.


In [ ]:
if CHE_DO_CHAY == 'preflight':
    assert (RUN/'cpu_preflight_pass.json').exists()
    report_zip=THU_MUC_KET_QUA/(RUN.name+'_CPU_PREFLIGHT.zip')
    with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in RUN.rglob('*'):
            if path.is_file() and path.suffix in {'.json','.csv'}:
                archive.write(path,str(Path(RUN.name)/path.relative_to(RUN)))
    print('CPU PREFLIGHT PASSED — 5528/4721; development4422/1106; chưa train/chưa tính điểm test.')
    print('Báo cáo nhỏ:',report_zip)
